<a href="https://colab.research.google.com/github/KayoLage/Ferramenta-SoftPipeline-INF450/blob/main/SoftPipe_Tool_INF450_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trabalho INF450 - Ferramenta didática para Soft. Pipeline**

## Ferramente Didática

### Exemplo usado para debug
```
loop: ld f1,0(r1)
mult f2,f1,f1
ld f3,4(r1)
mult f3,f3,f2
mult f3,f3,f1
add f3,f3,f2
sd f3,0(r1)
addi r1,r1,8
bne r1,r2,loop
```

### Imports

In [96]:
import re
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyArrowPatch
from matplotlib.patches import ConnectionStyle
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
from collections import defaultdict

from matplotlib._pylab_helpers import Gcf
Gcf.figs.clear()

### *Parsing Assembly* $\rightarrow$ *Python*

In [97]:
INSTR_TABLE = {
    "mult": ("r_type", "*"), "add":  ("r_type", "+"), "sub":  ("r_type", "-"),
    "addi": ("i_type", "+"), "subi": ("i_type", "-"),
    "ld": ("load", None), "sd":  ("store", None),
    "bne": ("branch", "!="), "beq": ("branch", "=="), "blt": ("branch", "<"),
    "bgt": ("branch", ">"), "ble": ("branch", "<="), "bge": ("branch", ">="),
    "j": ("jump", None),
    "mov": ("mov", None)
}
MEM_RE = re.compile(r"^(-?\d+)\((\w+)\)$")

def _reg_name(token):
    m = re.match(r"^([fr])(\d+)$", token)
    if m:
        bank, idx = m.groups()
        return f"{bank}[{idx}]"
    return token

def parse_line(raw_line, line_num=None):
    line = raw_line.split("#")[0].strip()
    if not line:
        return None

    label = None
    first_token = line.split()[0] if line.split() else ""
    if first_token.endswith(":"):
        label = first_token[:-1]
        line = line[len(first_token):].strip()
        if not line:
            return {"type": "label", "label": label, "raw": raw_line.strip(),
                    "python": f"# label: {label}"}

    m = re.match(r"^(\S+)\s+(.*)$", line)
    if not m:
        raise ValueError(f"Linha {line_num}: não consegui interpretar '{raw_line.strip()}'")
    opcode, rest = m.groups()

    if opcode not in INSTR_TABLE:
        raise ValueError(f"instrução desconhecida '{opcode}'")

    operands = [op.strip() for op in rest.split(",")]
    category, op = INSTR_TABLE[opcode]
    result = {"opcode": opcode, "category": category, "op": op, "label": label,
              "raw": raw_line.strip(), "operands": operands}

    expected_operands = {
        "r_type": 3, "i_type": 3, "load": 2, "store": 2, "branch": 3, "jump": 1, "mov": 2
    }
    if len(operands) != expected_operands[category]:
        raise ValueError(f"'{opcode}' espera {expected_operands[category]} operando(s), recebeu {len(operands)}")

    if category == "r_type":
        dest, src1, src2 = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src1)} {op} {_reg_name(src2)}"
    elif category == "i_type":
        dest, src, imm = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src)} {op} {imm}"
    elif category == "load":
        dest, mem = operands
        mm = MEM_RE.match(mem)
        if not mm: raise ValueError(f"endereço de memória inválido '{mem}'")
        offset, base = mm.groups()
        result["python"] = f"{_reg_name(dest)} = mem[{_reg_name(base)} + {offset}]"
    elif category == "store":
        src, mem = operands
        mm = MEM_RE.match(mem)
        if not mm: raise ValueError(f"endereço de memória inválido '{mem}'")
        offset, base = mm.groups()
        result["python"] = f"mem[{_reg_name(base)} + {offset}] = {_reg_name(src)}"
    elif category == "mov":
        dest, src = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src)}"
    elif category == "branch":
        src1, src2, target = operands
        result["python"] = f"if {_reg_name(src1)} {op} {_reg_name(src2)}: goto('{target}')"
    elif category == "jump":
        (target,) = operands
        result["python"] = f"goto('{target}')"

    if label:
        result["python"] = f"# {label}:\n" + result["python"]
    return result

def parse_program(text):
    out = []
    for i, raw_line in enumerate(text.split("\n"), 1):
        parsed = parse_line(raw_line, line_num=i)
        if parsed:
            out.append(parsed)
    return out


# =========================================================================
# ⚙️ MachineConfig — Configurações Fixas e Parâmetro de Loops Centralizado
# =========================================================================
class MachineConfig:
    def __init__(self, mem_size=35, word_size=4, forced_loops=30):
        self.mem_size = mem_size
        self.word_size = word_size
        self.forced_loops = forced_loops

        # Gera automaticamente r1=1, r2=2... e f1=1.0, f2=2.0... até 31
        self.reg_init = {}
        for i in range(1, 32):
            self.reg_init[f"r{i}"] = i
            self.reg_init[f"f{i}"] = float(i)

    def fresh_memory(self):
        return [float(x) for x in range(self.mem_size)]

    def fresh_registers(self):
        return dict(self.reg_init)

    def __repr__(self):
        return f"MachineConfig(mem_size={self.mem_size}, word_size={self.word_size}, forced_loops={self.forced_loops})"

### Ferramenta de Interação  

In [98]:
custom_css = widgets.HTML("""
<style>
.header-card {
    background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%);
    padding: 18px 24px; border-radius: 14px 14px 0 0; color: white; font-family: 'Segoe UI', sans-serif;
}
.header-card h3 { margin: 0; font-size: 20px; font-weight: 600; }
.header-card p { margin: 4px 0 0 0; font-size: 13px; opacity: 0.85; }
.tool-card { border: 1px solid #dcdde1; border-radius: 14px; padding: 0 0 18px 0; background: #fafbfc; box-shadow: 0 4px 16px rgba(0,0,0,0.08); margin-bottom: 16px; }
.inner-content { padding: 18px 24px 0 24px; }
.widget-textarea textarea { border-radius: 10px !important; border: 1.5px solid #a8dadc !important; font-family: 'Consolas','Courier New',monospace !important; font-size: 20px !important; line-height: 1.5 !important; padding: 12px !important; }
.widget-button { border-radius: 8px !important; font-weight: 600 !important; font-size: 13px !important; height: 38px !important; }
.output-box { background: white; border-radius: 10px; border: 1px solid #e0e0e0; padding: 8px 14px; font-family: 'Consolas','Courier New',monospace; font-size: 15px; color: #1d1d1d !important; }
.output-box pre { color: #1d1d1d !important; background: transparent !important; }
.output-box * { color: #1d1d1d !important; }
.config-panel { background: #eef2f7; border: 1px solid #dcdde1; border-radius: 10px; padding: 14px 20px; margin: 10px 0; font-family: 'Segoe UI', sans-serif; }
</style>
""")

# ---- estado do programa e da máquina fixa (compartilhados com a Célula 3) ----
program_instructions = []
machine_config = MachineConfig(mem_size=200, word_size=4) # TOMAR CUIDADO AQUI COM O TAMANHO DE MEMORIA
                                                          # A DEPENDER DE COMO SEU PROGRAMA ACESSA A MEMORIA SEJA
                                                          # COM LOADs ou STOREs EM INDICES QUE O TAMANHO DA MEMORIA
                                                          # INSTANCIADA PELO CONSTRUTOR DA CLASSE NAO SUPORTA

# ---------------- Widgets de entrada do programa ----------------
input_area = widgets.Textarea(
    value=(
        'loop: ld f1, 0(r1)\n'
        'mult f2, f1, f1\n'
        'ld f3, 4(r1)\n'
        'mult f3, f3, f2\n'
        'mult f3, f3, f1\n'
        'add f3, f3, f2\n'
        'sd f3, 0(r1)\n'
        'addi r1, r1, 8\n'
        'bne r1, r2, loop'
    ),
    description='',
    layout=widgets.Layout(width='95%', height='320px', margin='0 0 12px 0')
)

add_button = widgets.Button(description='Adicionar ao programa', icon='plus', button_style='success', layout=widgets.Layout(width='220px'))
clear_button = widgets.Button(description='Limpar programa', icon='trash', button_style='danger', layout=widgets.Layout(width='180px'))
graph_button = widgets.Button(description='Gerar grafo de dependências', icon='project-diagram', button_style='info', layout=widgets.Layout(width='260px'))

output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0', max_height='300px', overflow='auto'))
graph_output = widgets.Output(layout=widgets.Layout(margin='10px 0 0 0'))

def on_add_clicked(b):
    with output:
        clear_output()
        try:
            novas = parse_program(input_area.value)
            program_instructions.extend(novas)
            print(f"✅ {len(novas)} instrução(ões) adicionada(s). Total no programa: {len(program_instructions)}\n")
            for i, instr in enumerate(program_instructions):
                print(f"[{i}] {instr['raw']}  ->  {instr.get('python', '<<< SEM TRADUÇÃO >>>')}")
        except ValueError as e:
            print(f"❌ Erro de parsing: {e}")

def on_clear_clicked(b):
    global program_instructions
    program_instructions = []
    input_area.value = ''
    with output:
        clear_output(); print("🗑️ Programa limpo.")
    with graph_output:
        clear_output()

def extract_regs(raw_text):
    text = raw_text.lower().split('#')[0].strip()
    parts = text.split(None, 1)
    if len(parts) < 2:
        return set(), set()
    opcode, rest = parts
    operands = [op.strip() for op in rest.split(',')]
    find_f = lambda t: set(re.findall(r'f\d+', t))
    writes, reads = set(), set()
    if opcode == 'ld':
        writes |= find_f(operands[0]);
        if len(operands) > 1: reads |= find_f(operands[1])
    elif opcode == 'sd':
        reads |= find_f(operands[0])
        if len(operands) > 1: reads |= find_f(operands[1])
    elif opcode == 'mov':
        writes |= find_f(operands[0])
        if len(operands) > 1: reads |= find_f(operands[1])
    elif opcode in ('mult', 'add', 'sub'):
        writes |= find_f(operands[0])
        if len(operands) > 1: reads |= find_f(operands[1])
        if len(operands) > 2: reads |= find_f(operands[2])
    return writes, reads

def build_dependency_graph(instructions):
    G = nx.DiGraph()
    for i, instr in enumerate(instructions):
        raw = instr['raw']
        first_token = raw.split()[0] if raw.split() else ''
        if first_token.endswith(':'):
            raw = raw[len(first_token):].strip()
        G.add_node(i, text=raw)

    last_write, last_reads, edges = {}, defaultdict(list), defaultdict(set)
    for j, instr in enumerate(instructions):
        if instr.get('type') == 'label' or instr['category'] in ('branch', 'jump'):
            continue
        raw = instr['raw']
        first_token = raw.split()[0] if raw.split() else ''
        if first_token.endswith(':'):
            raw = raw[len(first_token):].strip()
        writes_j, reads_j = extract_regs(raw)

        for r in reads_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
        for r in writes_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
            for reader in last_reads.get(r, []): edges[(reader, j)].add(r)
        for r in reads_j: last_reads[r].append(j)
        for r in writes_j: last_write[r] = j; last_reads[r] = []

    for (u, v), regs in edges.items():
        G.add_edge(u, v, regs=regs)
    return G

def compute_layered_positions(G):
    level = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        level[node] = 0 if not preds else max(level[p] for p in preds) + 1
    return level

def on_graph_clicked(b):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Ellipse, FancyArrowPatch, ConnectionStyle
    with graph_output:
        clear_output()
        if not program_instructions:
            print("⚠️ Nenhuma instrução no programa ainda. Adicione instruções primeiro.")
            return
        G = build_dependency_graph(program_instructions)
        no_dep = [n for n in G.nodes() if G.degree(n) == 0]
        G.remove_nodes_from(no_dep)
        if G.number_of_nodes() == 0:
            print("ℹ️ Nenhuma instrução com dependência de registradores 'f' para exibir.")
            return

        level = compute_layered_positions(G)
        levels = defaultdict(list)
        for node, lvl in level.items(): levels[lvl].append(node)
        pos = {}
        x_gap, y_gap = 4.2, 2.8
        for lvl in sorted(levels):
            nodes_sorted = sorted(levels[lvl]); n = len(nodes_sorted)
            for i, node in enumerate(nodes_sorted):
                pos[node] = ((i - (n - 1) / 2) * x_gap, -lvl * y_gap)

        max_level = max(level.values())
        width = max(10, max(defaultdict(int, {l: len(v) for l, v in levels.items()}).values()) * 3.4)
        height = max(7, (max_level + 1) * 2.8)
        fig, ax = plt.subplots(figsize=(width, height), facecolor='white')

        node_boxes = {}
        for node, (x, y) in pos.items():
            text = G.nodes[node]['text']
            w = 1.0 + 0.16 * len(text); h = 0.95
            ax.add_patch(Ellipse((x, y), w, h, facecolor='#f1faee', edgecolor='#1d3557', linewidth=2, zorder=2))
            ax.text(x, y, text, ha='center', va='center', fontsize=11, fontfamily='monospace', fontweight='bold', color='#1d3557', zorder=3)
            node_boxes[node] = (x, y, w, h)

        xs = [p[0] for p in pos.values()]; min_x, max_x = min(xs) - 2, max(xs) + 2

        for lvl in range(1, max_level + 1):
            y_sep = -(lvl - 0.5) * y_gap
            ax.plot([min_x - 0.5, max_x + 0.5], [y_sep, y_sep], color='#b2bec3', linestyle='--', linewidth=1.5, zorder=0)
            ax.text(min_x - 0.8, y_sep, f"N={lvl}", fontsize=13, fontweight='bold', color='#2d3436', ha='right', va='center')

        for u, v, data in G.edges(data=True):
            x1, y1, w1, h1 = node_boxes[u]; x2, y2, w2, h2 = node_boxes[v]
            dx, dy = x2 - x1, y2 - y1; dist = (dx**2 + dy**2)**0.5 or 1
            ux, uy = dx / dist, dy / dist
            start = (x1 + ux * w1/2*0.9, y1 + uy * h1/2*0.9)
            end = (x2 - ux * w2/2*0.9, y2 - uy * h2/2*0.9)
            level_diff = level[v] - level[u]
            rad_value = (0.22 + 0.08 * level_diff) * (1 if (u + v) % 2 == 0 else -1) if level_diff > 1 else 0.0
            ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=16, color='#e63946', linewidth=1.7, zorder=1, connectionstyle=f'arc3,rad={rad_value}', shrinkA=0, shrinkB=0))
            verts = ConnectionStyle.Arc3(rad=rad_value).connect(start, end).vertices
            if len(verts) >= 3:
                P0, Pc, P2 = verts[0], verts[1], verts[-1]
                mx, my = 0.25*P0[0]+0.5*Pc[0]+0.25*P2[0], 0.25*P0[1]+0.5*Pc[1]+0.25*P2[1]
            else:
                mx, my = (verts[0][0]+verts[-1][0])/2, (verts[0][1]+verts[-1][1])/2
            ax.text(mx, my, ", ".join(sorted(data['regs'])), fontsize=16, color='#e63946', fontweight='bold', ha='center', va='center', zorder=4, bbox=dict(facecolor='white', edgecolor='none', pad=1.5))

        ax.set_xlim(min_x - 2, max_x + 2); ax.set_ylim(min(p[1] for p in pos.values()) - 2, 2)
        ax.axis('off'); ax.set_aspect('equal')
        plt.title("Grafo de Dependência de Registradores", fontsize=15, fontweight='bold', color='#1d3557')
        plt.tight_layout(); plt.show()

add_button.on_click(on_add_clicked)
clear_button.on_click(on_clear_clicked)
graph_button.on_click(on_graph_clicked)

output.add_class('output-box')
graph_output.add_class('output-box')

header = widgets.HTML("""
<div class="header-card">
    <h3>🔧 Editor de Instruções — Programa + Configuração da Máquina</h3>
    <p>Digite instruções Assembly, adicione ao programa e gere o grafo de dependências.</p>
</div>
""")

# Painel Informativo Fixo (Substitui os inputs interativos antigos)
config_panel = widgets.HTML(f"""
<div class="config-panel">
    <span style="color: #1d3557; font-weight: bold; font-size: 14px;">⚙️ Configuração Fixa da Máquina:</span><br/>
    <span style="font-size: 13px; color: #2d3436;">
        • <b>Tamanho da Memória:</b> {machine_config.mem_size} posições |
        • <b>Word Size:</b> {machine_config.word_size} bytes<br/>
        • <b>Estado Inicial dos Registradores:</b> <code style="background: #ffffff; padding: 2px 4px; border-radius: 4px;">r1=1, r2=2, ..., r31=31</code> e <code style="background: #ffffff; padding: 2px 4px; border-radius: 4px;">f1=1.0, f2=2.0, ..., f31=31.0</code>
    </span>
</div>
""")

body = widgets.VBox([
    input_area,
    widgets.HBox([add_button, clear_button, graph_button], layout=widgets.Layout(gap='10px')),
    config_panel,
    output,
    graph_output
], layout=widgets.Layout())
body.add_class('inner-content')

card = widgets.VBox([header, body])
card.add_class('tool-card')

display(custom_css, card)

HTML(value="\n<style>\n.header-card {\n    background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%);\n   …

### Gera grafo de soft. pipeline e compara com original

#### Usuário Propõe Grafo

In [115]:
from collections import defaultdict

def _ensure_mem_size(mem_list, idx):
    """Auxiliar para expandir a memória dinamicamente caso o índice vá além do limite atual"""
    if idx >= len(mem_list):
        mem_list.extend([float(i) for i in range(len(mem_list), idx + 1)])

# =========================================================================
# 🧠 ISASimulator — Versão Robusta com Memória Dinâmica
# =========================================================================
class ISASimulator:
    def __init__(self, config: MachineConfig):
        self.config = config

    @staticmethod
    def _get_reg(name, R, F):
        return F.setdefault(name, 0.0) if name.startswith('f') else R.setdefault(name, 0)

    @staticmethod
    def _set_reg(name, val, R, F):
        if name.startswith('f'):
            F[name] = val
        else:
            R[name] = val

    def execute_straightline(self, instr, R, F, mem):
        """Executa UMA instrução não-branch/jump/label com auto-expansão de memória."""
        cat, ops, op = instr['category'], instr['operands'], instr['op']
        ws = self.config.word_size

        if cat == 'r_type':
            dest, s1, s2 = ops
            a, b = self._get_reg(s1, R, F), self._get_reg(s2, R, F)
            val = a * b if op == '*' else (a + b if op == '+' else a - b)
            self._set_reg(dest, val, R, F)
        elif cat == 'i_type':
            dest, src, imm = ops
            a = self._get_reg(src, R, F)
            val = a + int(imm) if op == '+' else a - int(imm)
            self._set_reg(dest, val, R, F)
        elif cat == 'load':
            dest, mem_op = ops
            offset, base = MEM_RE.match(mem_op).groups()
            addr = self._get_reg(base, R, F) + int(offset)
            idx = addr // ws
            _ensure_mem_size(mem, idx)  # Expansão dinâmica na leitura
            self._set_reg(dest, mem[idx], R, F)
        elif cat == 'store':
            src, mem_op = ops
            offset, base = MEM_RE.match(mem_op).groups()
            addr = self._get_reg(base, R, F) + int(offset)
            idx = addr // ws
            _ensure_mem_size(mem, idx)  # Expansão dinâmica na escrita
            mem[idx] = self._get_reg(src, R, F)
        elif cat == 'mov':
            dest, src = ops
            self._set_reg(dest, self._get_reg(src, R, F), R, F)
        else:
            raise ValueError(f"categoria '{cat}' não é executável diretamente.")

    @staticmethod
    def build_label_map(instructions):
        labels = {}
        for i, instr in enumerate(instructions):
            if instr.get('type') == 'label' or instr.get('label'):
                labels[instr['label']] = i
        return labels

    def run_full_program(self, instructions, R_init=None, F_init=None, mem=None, max_steps=500_000):
        R = dict(R_init) if R_init is not None else self.config.fresh_registers()
        F = dict(F_init) if F_init is not None else {}
        mem = mem if mem is not None else self.config.fresh_memory()
        labels = self.build_label_map(instructions)
        pc, steps = 0, 0
        label_visits = defaultdict(int)
        n = len(instructions)

        while 0 <= pc < n and steps < max_steps:
            instr = instructions[pc]
            steps += 1

            if instr.get('type') == 'label':
                pc += 1
                continue

            if instr.get('label'):
                label_visits[instr['label']] += 1

            cat = instr['category']
            if cat == 'branch':
                s1, s2, target = instr['operands']
                a, b, op = self._get_reg(s1, R, F), self._get_reg(s2, R, F), instr['op']
                taken = {'!=': a != b, '==': a == b, '<': a < b,
                         '>': a > b, '<=': a <= b, '>=': a >= b}[op]

                if taken and target in label_visits and label_visits[target] >= self.config.forced_loops:
                    taken = False

                pc = labels[target] if taken else pc + 1

            elif cat == 'jump':
                (target,) = instr['operands']
                if target in label_visits and label_visits[target] >= self.config.forced_loops:
                    pc = pc + 1
                else:
                    pc = labels[target]
            else:
                self.execute_straightline(instr, R, F, mem)
                pc += 1

        return R, F, mem, label_visits

    @staticmethod
    def detect_induction_strides(instructions):
        strides = {}
        for instr in instructions:
            if instr['category'] == 'i_type':
                dest, src, imm = instr['operands']
                if dest == src and not dest.startswith('f'):
                    strides[dest] = int(imm) if instr['op'] == '+' else -int(imm)
        return strides

    @staticmethod
    def collect_written_regs(instructions):
        written = set()
        for instr in instructions:
            if instr.get('category') in ('r_type', 'i_type', 'load', 'mov'):
                written.add(instr['operands'][0])
        return written


# =========================================================================
# ✅ PipelineValidator — Versão com Suporte à Expansão de Memória
# =========================================================================
class PipelineValidator:
    def __init__(self, original_instructions, config: MachineConfig):
        self.original_instructions = original_instructions
        self.config = config
        self.simulator = ISASimulator(config)

    def compute_reference(self):
        _, _, mem_final, label_visits = self.simulator.run_full_program(self.original_instructions)
        strides = self.simulator.detect_induction_strides(self.original_instructions)
        total_iterations = max(label_visits.values()) if label_visits else 1
        return mem_final, strides, total_iterations

    def simulate_pipeline(self, stages_user, strides, total_iterations):
        ws = self.config.word_size
        mem_pipe = self.config.fresh_memory()
        F_pipe = {}
        max_lvl = max(stages_user.keys())
        total_cycles = total_iterations + max_lvl

        for cycle in range(total_cycles):
            F_snap = dict(F_pipe)
            mem_snap = list(mem_pipe)
            pending_F, pending_mem = {}, {}

            for lvl in sorted(stages_user.keys(), reverse=True):
                it = cycle - lvl
                if not (0 <= it < total_iterations):
                    continue
                R_it = {reg: self.config.reg_init.get(reg, 0) + it * step for reg, step in strides.items()}
                for instr in stages_user[lvl]:
                    cat, ops, op = instr['category'], instr['operands'], instr['op']
                    if cat == 'load':
                        dest, mem_op = ops
                        offset, base = MEM_RE.match(mem_op).groups()
                        addr = R_it.get(base, 0) + int(offset)
                        idx = addr // ws

                        if idx >= len(mem_pipe):
                            _ensure_mem_size(mem_pipe, idx)
                        val = mem_snap[idx] if idx < len(mem_snap) else float(idx)
                        pending_F[dest] = val
                    elif cat == 'store':
                        src, mem_op = ops
                        offset, base = MEM_RE.match(mem_op).groups()
                        addr = R_it.get(base, 0) + int(offset)
                        idx = addr // ws
                        if idx >= len(mem_pipe):
                            _ensure_mem_size(mem_pipe, idx)
                        pending_mem[idx] = F_snap.get(src, pending_F.get(src, 0.0))
                    elif cat == 'mov':
                        dest, src = ops
                        pending_F[dest] = F_snap.get(src, pending_F.get(src, 0.0))
                    elif cat == 'r_type':
                        dest, s1, s2 = ops
                        a = F_snap.get(s1, pending_F.get(s1, 0.0))
                        b = F_snap.get(s2, pending_F.get(s2, 0.0))
                        pending_F[dest] = a * b if op == '*' else (a + b if op == '+' else a - b)
                    elif cat == 'i_type':
                        dest, src, imm = ops
                        a = F_snap.get(src, pending_F.get(src, 0.0)) if src.startswith('f') else R_it.get(src, 0)
                        pending_F[dest] = a + int(imm) if op == '+' else a - int(imm)
                    else:
                        raise ValueError(f"categoria '{cat}' não suportada dentro do corpo do pipeline")

            F_pipe.update(pending_F)
            for idx, val in pending_mem.items():
                mem_pipe[idx] = val

        return mem_pipe

    def validate(self, user_canvas_nodes):
        if not user_canvas_nodes:
            return {"error": "O canvas está vazio! Monte um grafo antes de validar."}
        if not self.original_instructions:
            return {"error": "Nenhum programa original carregado para servir de referência."}

        try:
            mem_gabarito, strides, total_iterations = self.compute_reference()
        except Exception as e:
            return {"error": f"Erro executando o programa original: {e}"}

        stages_user = defaultdict(list)
        for item in user_canvas_nodes:
            stages_user[item['level']].append(parse_line(item['text'], 0))

        try:
            mem_pipe = self.simulate_pipeline(stages_user, strides, total_iterations)
        except Exception as e:
            return {"error": f"Erro na simulação do pipeline: {e}"}

        return {
            "success": mem_gabarito == mem_pipe,
            "mem_gabarito": mem_gabarito,
            "mem_pipe": mem_pipe,
            "strides": strides,
            "total_iterations": total_iterations,
            "error": None,
        }

# =========================================================================
# 🎨 Componentes de Interface do Canvas (Modificado)
# =========================================================================
canvas_title = widgets.HTML("""
    <style>
    .no-pointer-plots canvas, .no-pointer-plots .jupyter-matplotlib { pointer-events: none !important; }
    </style>
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;">🎨 Canvas Interativo: Proponha o Grafo de Software Pipelining</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Monte o grafo movendo instruções para as camadas ou escreva o grafo por extenso. A configuração da máquina (registradores/memória) é a definida na Célula 2.</p>
    </div>
""")

txt_node_instruction = widgets.Text(value='mov f4, f1', placeholder='Ex: mult f2, f1, f1', description='Instrução:', layout=widgets.Layout(width='280px'))
slider_node_level = widgets.IntSlider(value=0, min=0, max=8, step=1, description='Nível (N):', layout=widgets.Layout(width='240px'))
btn_add_canvas_node = widgets.Button(description='Adicionar Nó', icon='plus', button_style='success', layout=widgets.Layout(width='140px', height='34px'))

txt_graph_code = widgets.Textarea(
    value='# Formato esperado: [nivel] inst\n[0] ld f1, 0(r1)\n[0] ld f3, 4(r1)\n[1] mov f4, f1\n[1] mult f2, f1, f1\n[1] mov f5, f3\n[2] mov f6, f4\n[2] mov f7, f2\n[2] mult f8, f5, f2\n[3] mov f9, f7\n[3] mult f10, f8, f6\n[4] add f11, f10, f9\n[5] sd f11, 0(r1)',
    placeholder='Digite o grafo por extenso...',
    description='Script Grafo:',
    layout=widgets.Layout(width='675px', height='125px', margin='5px 0 10px 0')
)
btn_load_graph_code = widgets.Button(description='Carregar Grafo por Escrito', icon='code', button_style='info', layout=widgets.Layout(width='230px', height='34px', margin='40px 0 0 0'))

drop_remove_node = widgets.Dropdown(options=[('Nenhum nó disponível', -1)], description='Selecionar Nó:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'), disabled=True)
btn_remove_canvas_node = widgets.Button(description='Remover Selecionado', icon='trash', button_style='warning', layout=widgets.Layout(width='180px', height='34px'))
btn_clear_canvas = widgets.Button(description='Limpar Canvas', icon='trash-restore', button_style='danger', layout=widgets.Layout(width='140px', height='34px'))
btn_validate_canvas_graph = widgets.Button(description='Validar Grafo Proposto', icon='shield-check', button_style='primary', layout=widgets.Layout(width='220px', height='34px'))

canvas_output_plot = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))
canvas_output_plot.add_class('no-pointer-plots')

user_canvas_nodes = []

def update_remove_dropdown():
    if not user_canvas_nodes:
        drop_remove_node.options = [('Nenhum nó disponível', -1)]; drop_remove_node.disabled = True
    else:
        drop_remove_node.options = [(f"[{idx}] {item['text']} (Nível {item['level']})", idx) for idx, item in enumerate(user_canvas_nodes)]
        drop_remove_node.disabled = False

def build_user_proposed_graph(nodes_list):
    import networkx as nx
    G = nx.DiGraph()
    for idx, item in enumerate(nodes_list): G.add_node(idx, text=item['text'], level=item['level'])
    last_write, last_reads, edges = {}, defaultdict(list), defaultdict(set)
    for j, item in enumerate(nodes_list):
        writes_j, reads_j = extract_regs(item['text'])
        for r in reads_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
        for r in writes_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
            for reader in last_reads.get(r, []): edges[(reader, j)].add(r)
        for r in reads_j: last_reads[r].append(j)
        for r in writes_j: last_write[r] = j; last_reads[r] = []
    for (u, v), regs in edges.items(): G.add_edge(u, v, regs=regs)
    return G

def compute_positions_barycenter_local(G, levels_dict):
    levels = defaultdict(list)
    for node, lvl in levels_dict.items(): levels[lvl].append(node)
    pos = {}
    x_gap, y_gap = 7.5, 4.2; sorted_levels = sorted(levels.keys())
    if sorted_levels:
        min_lvl = sorted_levels[0]
        for i, node in enumerate(sorted(levels[min_lvl])): pos[node] = ((i - (len(levels[min_lvl]) - 1) / 2) * x_gap, -min_lvl * y_gap)
    for lvl in sorted_levels[1:]:
        node_barycenters = {}
        for node in levels[lvl]:
            parents = list(G.predecessors(node))
            node_barycenters[node] = sum(pos[p][0] for p in parents) / len(parents) if parents else 0
        n = len(levels[lvl])
        for i, node in enumerate(sorted(levels[lvl], key=lambda n: node_barycenters[n])): pos[node] = ((i - (n - 1) / 2) * x_gap, -lvl * y_gap)
    return pos

def compute_positions_user_canvas(nodes_list):
    levels = defaultdict(list)
    for idx, item in enumerate(nodes_list): levels[item['level']].append(idx)
    pos = {}
    x_gap, y_gap = 7.5, 4.2
    for lvl in sorted(levels.keys()):
        nodes_in_lvl = sorted(levels[lvl]); n = len(nodes_in_lvl)
        for i, node_idx in enumerate(nodes_in_lvl): pos[node_idx] = ((i - (n - 1) / 2) * x_gap, -lvl * y_gap)
    return pos

def draw_graph_on_axis(ax, G, pos, levels_dict, node_text_map, dest_changed_map, is_pipelined, x_limits, y_limits):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Ellipse, FancyArrowPatch, ConnectionStyle
    from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
    if not levels_dict: return

    def get_text_tokens_and_colors(text, dest_changed):
        text = text.lower().strip()
        parts = text.split(None, 1)
        if len(parts) == 2:
            opcode, rest = parts; subparts = rest.split(',', 1)
            if len(subparts) == 2:
                dest, remaining = subparts; color_dest = "red" if dest_changed else "#1e293b"
                return [opcode + " ", dest.strip(), ", " + remaining.strip()], ["#1e293b", color_dest, "#1e293b"]
            else:
                if opcode.lower() in ['sd', 'sw', 'sb', 'sh']: return [text], ["#1e293b"]
                return [opcode + " ", rest], ["#1e293b", "red" if dest_changed else "#1e293b"]
        return [text], ["#1e293b"]

    node_boxes, y_gap = {}, 4.2
    for node in G.nodes():
        x, y = pos[node]; text = node_text_map[node]; w = 1.4 + 0.26 * max(len(t) for t in text.split('\n')); h = 1.1
        ax.add_patch(Ellipse((x, y), w, h, facecolor='white', edgecolor='#1e293b', linewidth=2, zorder=2))
        if is_pipelined and dest_changed_map:
            tokens, colors = get_text_tokens_and_colors(text, dest_changed_map[node])
            children = [TextArea(t, textprops=dict(color=c, fontweight='bold', fontfamily='monospace', fontsize=13)) for t, c in zip(tokens, colors)]
            packer = HPacker(children=children, align="center", pad=0, sep=0)
            ab = AnnotationBbox(packer, (x, y), xycoords='data', box_alignment=(0.5, 0.5), bboxprops=dict(facecolor='none', edgecolor='none'), pad=0, zorder=3)
            ax.add_artist(ab)
        else:
            ax.text(x, y, text, ha='center', va='center', fontsize=13, fontfamily='monospace', fontweight='bold', color='#1e293b', zorder=3)
        node_boxes[node] = (x, y, w, h)

    min_x_global, max_x_global = x_limits; min_level, max_level = min(levels_dict.values()), max(levels_dict.values())
    for lvl in range(min_level, max_level + 1):
        y_sep = -(lvl - 0.5) * y_gap
        if lvl > min_level: ax.plot([min_x_global - 0.5, max_x_global + 0.5], [y_sep, y_sep], color='#cbd5e1', linestyle='-', linewidth=1.5, zorder=0)
        ax.text(max_x_global + 0.8, -lvl * y_gap, f"{lvl}", fontsize=18, fontweight='bold', color='#475569', ha='left', va='center')

    for edge_idx, (u, v, data) in enumerate(G.edges(data=True)):
        x1, y1, _, _ = node_boxes[u]; x2, y2, _, _ = node_boxes[v]; level_diff = levels_dict[v] - levels_dict[u]
        rad_value = (-1 if (x1 + x2) / 2 <= 0 else 1) * 0.22 if level_diff > 1 else 0.0
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=15, color='#475569', linewidth=1.8, zorder=1, connectionstyle=f'arc3,rad={rad_value}', shrinkA=22, shrinkB=22))
        verts = ConnectionStyle.Arc3(rad=rad_value).connect((x1, y1), (x2, y2)).vertices
        mx, my = (verts[0][0] + verts[-1][0]) / 2, (verts[0][1] + verts[-1][1]) / 2
        ax.text(mx, my, ", ".join(sorted(data['regs'])), fontsize=11, color='#475569', fontweight='bold', ha='center', va='center', zorder=4, bbox=dict(facecolor='white', edgecolor='#cbd5e1', linewidth=1, boxstyle='round,pad=0.2'))

    ax.set_xlim(x_limits[0], x_limits[1]); ax.set_ylim(y_limits[0], y_limits[1]); ax.axis('off'); ax.set_aspect('equal')

def render_canvas_split_view():
    import matplotlib.pyplot as plt
    with canvas_output_plot:
        clear_output(wait=True)
        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)
        if G_orig.number_of_nodes() == 0:
            fig, ax = plt.subplots(figsize=(20, 3), facecolor='white')
            ax.text(0.5, 0.5, "⚠️ Aguardando carregamento de instruções válidas na Célula 2.", ha='center', va='center', fontsize=14, color='#64748b', fontweight='bold')
            ax.axis('off'); plt.show(); return

        level_orig = compute_layered_positions(G_orig)
        pos_orig = compute_positions_barycenter_local(G_orig, level_orig)
        node_text_map_orig = {n: G_orig.nodes[n]['text'] for n in G_orig.nodes()}

        original_written = ISASimulator.collect_written_regs(program_instructions)

        G_user = build_user_proposed_graph(user_canvas_nodes)
        level_user = {idx: item['level'] for idx, item in enumerate(user_canvas_nodes)}
        pos_user = compute_positions_user_canvas(user_canvas_nodes)
        node_text_map_user = {idx: item['text'] for idx, item in enumerate(user_canvas_nodes)}

        def is_renamed(text):
            parts = text.split(None, 1)
            if len(parts) < 2: return False
            opcode, rest = parts
            dest = rest.split(',')[0].strip()
            return opcode == 'mov' or dest not in original_written

        dest_changed_user = {idx: is_renamed(item['text']) for idx, item in enumerate(user_canvas_nodes)}

        all_xs = [p[0] for p in pos_orig.values()] + ([p[0] for p in pos_user.values()] if pos_user else [0])
        global_x_limits = (min(all_xs) - 4.5, max(all_xs) + 4.5); global_y_limits = (-24.0, 3.0)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 9.5), facecolor='white')
        ax1.set_navigate(False); ax2.set_navigate(False)

        draw_graph_on_axis(ax1, G_orig, pos_orig, level_orig, node_text_map_orig, None, False, global_x_limits, global_y_limits)
        ax1.set_title("1. Grafo de Dependência Original (Referência)", fontsize=16, fontweight='bold', color='#0f172a', pad=15)

        if user_canvas_nodes:
            draw_graph_on_axis(ax2, G_user, pos_user, level_user, node_text_map_user, dest_changed_user, True, global_x_limits, global_y_limits)
        else:
            for lvl in range(6):
                y_sep = -(lvl - 0.5) * 4.2
                ax2.plot([global_x_limits[0] - 0.5, global_x_limits[1] + 0.5], [y_sep, y_sep], color='#cbd5e1', linestyle='-')
                ax2.text(global_x_limits[1] + 0.8, -lvl * 4.2, f"{lvl}", fontsize=18, fontweight='bold', color='#94a3b8', ha='left', va='center')
            ax2.text(0, -10, "Canvas em Branco\n\nAdicione instruções utilizando\nos controles acima.", ha='center', va='center', color='#94a3b8', fontsize=16)
            ax2.set_xlim(global_x_limits[0], global_x_limits[1]); ax2.set_ylim(global_y_limits[0], global_y_limits[1]); ax2.axis('off'); ax2.set_aspect('equal')

        ax2.set_title("2. Seu Grafo Proposto (Software Pipelining)", fontsize=16, fontweight='bold', color='#0f172a', pad=15)
        fig.subplots_adjust(left=0.02, right=0.93, top=0.90, bottom=0.05, wspace=0.15)
        plt.show()

def on_validate_canvas_clicked(b):
    render_canvas_split_view()
    with canvas_output_plot:
        validator = PipelineValidator(program_instructions, machine_config)
        result = validator.validate(user_canvas_nodes)

        if result.get("error"):
            print(f"❌ {result['error']}")
            return

        n_show = min(30, machine_config.mem_size)

        if result["success"]:
            print("\n✅ VEREDICTO: SEU GRAFO DE PIPELINE ESTÁ CORRETO!")
            print("A distribuição de níveis e as dependências de registradores geraram equivalência semântica perfeita na memória.\n")
        else:
            print("\n❌ VEREDICTO: GRAFO INCORRETO (Hazard ou Renomeação Inválida)")
        print(f"    Gabarito Esperado : {result['mem_gabarito'][:n_show]}")
        print(f"    Sua Saída         : {result['mem_pipe'][:n_show]}")

def on_add_node_clicked(b):
    inst_text = txt_node_instruction.value.strip()
    if inst_text:
        try:
            parsed_result = parse_line(inst_text.lower(), line_num=len(user_canvas_nodes) + 1)
            if parsed_result:
                user_canvas_nodes.append({'text': inst_text.lower(), 'level': slider_node_level.value})
                update_remove_dropdown(); render_canvas_split_view()
        except ValueError as e:
            with canvas_output_plot: print(f"❌ REJEITADO PELO PARSER: {e}")

def on_load_graph_code_clicked(b):
    global user_canvas_nodes
    code_block = txt_graph_code.value.strip()
    if not code_block: return
    parsed_nodes = []
    try:
        for idx, line in enumerate(code_block.split('\n'), 1):
            line_clean = line.strip()
            if not line_clean or line_clean.startswith('#'): continue
            # O Regex aceita tanto "[nivel]" quanto "[nivel, id_nivel]" mantendo retrocompatibilidade
            match = re.match(r"^\[\s*(\d+)\s*(?:,\s*\d+\s*)?\]\s*(.+)$", line_clean)
            if not match: raise ValueError(f"Linha {idx}: Formato de tags inválido.")
            lvl_str, inst_raw = match.groups(); lvl = int(lvl_str); inst_raw = inst_raw.strip().lower()
            if parse_line(inst_raw, line_num=idx): parsed_nodes.append({'text': inst_raw, 'level': lvl})
        user_canvas_nodes = parsed_nodes
        update_remove_dropdown(); render_canvas_split_view()
        with canvas_output_plot: print(f"✅ Script carregado com sucesso! {len(user_canvas_nodes)} nós injetados no Canvas.")
    except ValueError as e:
        with canvas_output_plot: print(f"❌ ERRO AO PARSEAR SCRIPT DO GRAFO: {e}")

def on_remove_node_clicked(b):
    idx_to_remove = drop_remove_node.value
    if idx_to_remove is not None and idx_to_remove >= 0 and idx_to_remove < len(user_canvas_nodes):
        user_canvas_nodes.pop(idx_to_remove)
        update_remove_dropdown(); render_canvas_split_view()

def on_clear_canvas_clicked(b):
    global user_canvas_nodes; user_canvas_nodes = []
    update_remove_dropdown(); render_canvas_split_view()

btn_add_canvas_node.on_click(on_add_node_clicked)
btn_load_graph_code.on_click(on_load_graph_code_clicked)
btn_remove_canvas_node.on_click(on_remove_node_clicked)
btn_clear_canvas.on_click(on_clear_canvas_clicked)
btn_validate_canvas_graph.on_click(on_validate_canvas_clicked)

row1_creation = widgets.HBox([txt_node_instruction, slider_node_level, btn_add_canvas_node], layout=widgets.Layout(gap='15px', margin='0 0 5px 0'))
row2_bulk_code = widgets.HBox([txt_graph_code, btn_load_graph_code], layout=widgets.Layout(gap='15px', margin='0 0 10px 0'))
row3_actions = widgets.HBox([drop_remove_node, btn_remove_canvas_node, btn_clear_canvas, btn_validate_canvas_graph], layout=widgets.Layout(gap='15px'))
controls_form = widgets.VBox([row1_creation, row2_bulk_code, row3_actions], layout=widgets.Layout(padding='15px', background_color='#fafbfc', border='1px solid #cbd5e1', border_radius='0 0 12px 14px'))

canvas_dashboard = widgets.VBox([canvas_title, controls_form, canvas_output_plot])
display(canvas_dashboard)

update_remove_dropdown()
render_canvas_split_view()

#### Gabarito

In [104]:
# ---------------- Componentes de Interface Passo a Passo do Grafo ----------------
btn_graph_prev = widgets.Button(description='Passo Anterior', icon='arrow-left', button_style='warning', layout=widgets.Layout(width='160px', height='40px'))
btn_graph_next = widgets.Button(description='Próximo Passo', icon='arrow-right', button_style='success', layout=widgets.Layout(width='160px', height='40px'))
btn_graph_final = widgets.Button(description='Ir para o Final', icon='fast-forward', button_style='info', layout=widgets.Layout(width='160px', height='40px'))

lbl_graph_step = widgets.Label(value='Passo 0 de 0', layout=widgets.Layout(margin='8px 0 0 15px'))
lbl_graph_step.style.text_color = '#f1f5f9'

pipe_compare_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

# Variáveis globais de controle de passos do grafo
current_graph_idx = 0
graph_nodes_order = []

def get_text_tokens_and_colors(text, dest_changed):
    """Função auxiliar para colorir o registrador de destino caso ele tenha sido renomeado"""
    text = text.lower().strip()
    parts = text.split(None, 1)
    if len(parts) == 2:
        opcode, rest = parts; subparts = rest.split(',', 1)
        if len(subparts) == 2:
            dest, remaining = subparts; color_dest = "red" if dest_changed else "#1e293b"
            return [opcode + " ", dest.strip(), ", " + remaining.strip()], ["#1e293b", color_dest, "#1e293b"]
        else:
            if opcode.lower() in ['sd', 'sw', 'sb', 'sh']: return [text], ["#1e293b"]
            return [opcode + " ", rest], ["#1e293b", "red" if dest_changed else "#1e293b"]
    return [text], ["#1e293b"]

def extract_dest_reg(text):
    """Identifica o registrador de destino real da instrução (ignora stores/branches)"""
    text = text.split('#')[0].strip()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    if opcode.lower() in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def build_pipelined_with_moves_ordered(G_orig, level_orig):
    """Gera o grafo de pipeline guardando RIGOROSAMENTE a ordem de criação de cada nó"""
    all_regs = []
    for node in G_orig.nodes():
        all_regs.extend(re.findall(r'f\d+', G_orig.nodes[node]['text']))
    reg_indices = [int(r[1:]) for r in all_regs if r.startswith('f')]
    max_reg_idx = max(reg_indices) if reg_indices else 0
    global_next_reg = max_reg_idx + 1

    G_pipe = nx.DiGraph()
    level_pipe = {}
    node_text_map = {}
    dest_changed_map = {}
    nodes_order = []

    levels_to_nodes = defaultdict(list)
    for node, lvl in level_orig.items():
        levels_to_nodes[lvl].append(node)

    reg_name_at_level = defaultdict(dict)
    mov_nodes = {}
    move_node_counter = 1000
    seen_destinations = set()

    prod_consumers = defaultdict(list)
    for u, v, data in G_orig.edges(data=True):
        for r in data['regs']:
            prod_consumers[(u, r)].append(v)

    for u in G_orig.nodes():
        orig_text = G_orig.nodes[u]['text']
        d = extract_dest_reg(orig_text)
        if d:
            reg_name_at_level[(u, d)][level_orig[u]] = d

    for lvl in sorted(levels_to_nodes.keys()):
        # A. Processa e estende as cadeias de MOV nos mesmos níveis
        for (p, r), consumers in prod_consumers.items():
            L_p = level_orig[p]
            L_end = max(level_orig[c] for c in consumers)

            if L_p < lvl <= L_end:
                if lvl < L_end:
                    mov_id = move_node_counter
                    move_node_counter += 1

                    r_prev = reg_name_at_level[(p, r)][lvl - 1]
                    r_new = f"f{global_next_reg}"
                    global_next_reg += 1

                    node_text_map[mov_id] = f"mov {r_new}, {r_prev}"
                    level_pipe[mov_id] = lvl
                    dest_changed_map[mov_id] = True
                    G_pipe.add_node(mov_id)
                    nodes_order.append(mov_id)

                    mov_nodes[(p, r, lvl)] = mov_id
                    reg_name_at_level[(p, r)][lvl] = r_new

                    parent = p if lvl - 1 == L_p else mov_nodes[(p, r, lvl - 1)]
                    G_pipe.add_edge(parent, mov_id, regs={r_prev})
                else:
                    reg_name_at_level[(p, r)][lvl] = reg_name_at_level[(p, r)][lvl - 1]

        # B. Processa as instruções originais alocadas neste nível
        for u in levels_to_nodes[lvl]:
            orig_text = G_orig.nodes[u]['text']

            src_mappings = {}
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    src_mappings[r] = reg_name_at_level[(p_node, r)][lvl - 1]

            orig_dest = extract_dest_reg(orig_text)
            new_dest = orig_dest
            dest_changed = False

            if orig_dest:
                if orig_dest in seen_destinations:
                    new_dest = f"f{global_next_reg}"
                    global_next_reg += 1
                    dest_changed = True
                seen_destinations.add(orig_dest)

            parts = orig_text.split(None, 1)
            if len(parts) == 2:
                opcode, rest = parts
                subparts = rest.split(',')
                is_store = opcode.lower() in ['sd', 'sw', 'sb', 'sh']

                new_subparts = []
                for idx, sub in enumerate(subparts):
                    sub_clean = sub.strip()
                    if idx == 0 and not is_store and orig_dest is not None:
                        new_subparts.append(new_dest)
                    else:
                        updated_sub = sub_clean
                        for orig_src, mapped_name in src_mappings.items():
                            updated_sub = re.sub(r'\b' + re.escape(orig_src) + r'\b', mapped_name, updated_sub)
                        new_subparts.append(updated_sub)

                updated_text = f"{opcode} {', '.join(new_subparts)}"
            else:
                updated_text = orig_text

            node_text_map[u] = updated_text
            level_pipe[u] = lvl
            dest_changed_map[u] = dest_changed
            G_pipe.add_node(u)
            nodes_order.append(u)

            if orig_dest:
                reg_name_at_level[(u, orig_dest)][lvl] = new_dest

            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    parent_in_pipe = p_node if lvl - 1 == level_orig[p_node] else mov_nodes[(p_node, r, lvl - 1)]
                    r_name_used = reg_name_at_level[(p_node, r)][lvl - 1]
                    G_pipe.add_edge(parent_in_pipe, u, regs={r_name_used})

    return G_pipe, level_pipe, node_text_map, dest_changed_map, nodes_order

def draw_graph_incremental(ax, G, pos, levels_dict, node_text_map, dest_changed_map, is_pipelined, x_limits, y_limits, visible_nodes_set):
    """Desenha o grafo exibindo apenas os nós contidos no passo atual da animação"""
    node_boxes = {}
    y_gap = 4.2

    for node in G.nodes():
        if node not in visible_nodes_set: continue
        x, y = pos[node]
        text = node_text_map[node]
        w = 1.4 + 0.26 * max(len(t) for t in text.split('\n'))
        h = 1.1

        ellipse = Ellipse((x, y), w, h, facecolor='white', edgecolor='#1e293b', linewidth=2, zorder=2)
        ax.add_patch(ellipse)

        if is_pipelined:
            tokens, colors = get_text_tokens_and_colors(text, dest_changed_map[node])
            children = [TextArea(t, textprops=dict(color=c, fontweight='bold', fontfamily='monospace', fontsize=13))
                        for t, c in zip(tokens, colors)]
            packer = HPacker(children=children, align="center", pad=0, sep=0)
            ab = AnnotationBbox(packer, (x, y), xycoords='data', box_alignment=(0.5, 0.5),
                                bboxprops=dict(facecolor='none', edgecolor='none'), pad=0, zorder=3)
            ax.add_artist(ab)
        else:
            ax.text(x, y, text, ha='center', va='center', fontsize=13,
                    fontfamily='monospace', fontweight='bold', color='#1e293b', zorder=3)

        node_boxes[node] = (x, y, w, h)

    min_x_global, max_x_global = x_limits
    active_levels = [levels_dict[n] for n in visible_nodes_set]
    min_level = min(levels_dict.values()) if levels_dict else 0
    max_level = max(active_levels) if active_levels else min_level

    for lvl in range(min(levels_dict.values()), max(levels_dict.values()) + 1):
        if lvl > max_level: break
        y_sep = -(lvl - 0.5) * y_gap
        if lvl > min(levels_dict.values()):
            ax.plot([min_x_global - 0.5, max_x_global + 0.5], [y_sep, y_sep],
                    color='#cbd5e1', linestyle='-', linewidth=1.5, zorder=0)

        ax.text(max_x_global + 0.8, -lvl * y_gap, f"{lvl}", fontsize=18,
                fontweight='bold', color='#475569', ha='left', va='center')

    for edge_idx, (u, v, data) in enumerate(G.edges(data=True)):
        if u not in visible_nodes_set or v not in visible_nodes_set: continue

        x1, y1, _, _ = node_boxes[u]
        x2, y2, _, _ = node_boxes[v]

        u_lvl, v_lvl = levels_dict[u], levels_dict[v]
        level_diff = v_lvl - u_lvl

        avg_x = (x1 + x2) / 2
        direction = -1 if avg_x <= 0 else 1
        rad_value = direction * 0.22 if level_diff > 1 else 0.0

        inter_nodes = [n for n in visible_nodes_set if u_lvl < levels_dict[n] < v_lvl]

        if level_diff > 1:
            for _ in range(15):
                connector = ConnectionStyle.Arc3(rad=rad_value)
                shaft_path = connector.connect((x1, y1), (x2, y2))
                verts = shaft_path.vertices

                hit_node = False
                for t in [0.25, 0.5, 0.75]:
                    if len(verts) >= 3:
                        tx = (1-t)**2 * verts[0][0] + 2*(1-t)*t * verts[1][0] + t**2 * verts[-1][0]
                        ty = (1-t)**2 * verts[0][1] + 2*(1-t)*t * verts[1][1] + t**2 * verts[-1][1]
                    else:
                        tx = verts[0][0] + t * (verts[-1][0] - verts[0][0])
                        ty = verts[0][1] + t * (verts[-1][1] - verts[0][1])

                    for m in inter_nodes:
                        mx_n, my_n, wm_n, hm_n = node_boxes[m]
                        if abs(ty - my_n) < hm_n * 0.8 and abs(tx - mx_n) < (wm_n / 2 + 0.6):
                            hit_node = True
                            break
                    if hit_node: break

                if hit_node:
                    rad_value += direction * 0.15
                else:
                    break
        else:
            if abs(x1 - x2) < 0.1:
                rad_value = 0.15 * (1 if edge_idx % 2 == 0 else -1)

        arrow = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=15,
                                color='#475569', linewidth=1.8, zorder=1,
                                connectionstyle=f'arc3,rad={rad_value}', shrinkA=22, shrinkB=22)
        ax.add_patch(arrow)

        connector = ConnectionStyle.Arc3(rad=rad_value)
        shaft_path = connector.connect((x1, y1), (x2, y2))
        verts = shaft_path.vertices

        mx, my = (verts[0][0] + verts[-1][0])/2, (verts[0][1] + verts[-1][1])/2
        for t_pos in [0.35, 0.65, 0.5, 0.22, 0.78]:
            if len(verts) >= 3:
                tx = (1-t_pos)**2 * verts[0][0] + 2*(1-t_pos)*t_pos * verts[1][0] + t_pos**2 * verts[-1][0]
                ty = (1-t_pos)**2 * verts[0][1] + 2*(1-t_pos)*t_pos * verts[1][1] + t_pos**2 * verts[-1][1]
            else:
                tx = verts[0][0] + t_pos * (verts[-1][0] - verts[0][0])
                ty = verts[0][1] + t_pos * (verts[-1][1] - verts[0][1])

            collision = False
            for n_id in visible_nodes_set:
                nx_p, ny_p, nw_p, nh_p = node_boxes[n_id]
                if abs(tx - nx_p) < (nw_p/2 + 0.3) and abs(ty - ny_p) < (nh_p/2 + 0.3):
                    collision = True
                    break
            if not collision:
                mx, my = tx, ty
                break

        reg_label = ", ".join(sorted(data['regs']))
        ax.text(mx, my, reg_label, fontsize=11, color='#475569', fontweight='bold',
                ha='center', va='center', zorder=4,
                bbox=dict(facecolor='white', edgecolor='#cbd5e1', linewidth=1, boxstyle='round,pad=0.2'))

    ax.set_xlim(x_limits[0], x_limits[1])
    ax.set_ylim(y_limits[0], y_limits[1])
    ax.axis('off')
    ax.set_aspect('equal')

def compute_positions_barycenter(G, levels_dict):
    levels = defaultdict(list)
    for node, lvl in levels_dict.items(): levels[lvl].append(node)
    pos = {}
    x_gap, y_gap = 7.5, 4.2
    sorted_levels = sorted(levels.keys())
    if sorted_levels:
        min_lvl = sorted_levels[0]
        for i, node in enumerate(sorted(levels[min_lvl])):
            pos[node] = ((i - (len(levels[min_lvl]) - 1) / 2) * x_gap, -min_lvl * y_gap)
    for lvl in sorted_levels[1:]:
        node_barycenters = {}
        for node in levels[lvl]:
            parents = list(G.predecessors(node))
            node_barycenters[node] = sum(pos[p][0] for p in parents) / len(parents) if parents else 0
        n = len(levels[lvl])
        for i, node in enumerate(sorted(levels[lvl], key=lambda n: node_barycenters[n])):
            pos[node] = ((i - (n - 1) / 2) * x_gap, -lvl * y_gap)
    return pos

def render_current_graph_step():
    """Gera a plotagem lado a lado filtrando o progresso da pipeline nó por nó"""
    with pipe_compare_output:
        clear_output(wait=True)

        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)

        level_orig = compute_layered_positions(G_orig)
        G_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe, nodes_order = build_pipelined_with_moves_ordered(G_orig, level_orig)

        pos_orig = compute_positions_barycenter(G_orig, level_orig)
        pos_pipe = compute_positions_barycenter(G_pipe, level_pipe)

        visible_pipe_nodes = set(nodes_order[:current_graph_idx + 1])
        active_node_text = node_text_map_pipe[nodes_order[current_graph_idx]]

        lbl_graph_step.value = f"Passo {current_graph_idx + 1} de {len(nodes_order)} | Adicionando nó: [{active_node_text}]"

        all_xs = [p[0] for p in pos_orig.values()] + [p[0] for p in pos_pipe.values()]
        all_ys = [p[1] for p in pos_orig.values()] + [p[1] for p in pos_pipe.values()]
        global_x_limits = (min(all_xs) - 4.5, max(all_xs) + 4.5)
        global_y_limits = (min(all_ys) - 3.0, max(all_ys) + 3.0)

        max_level = max(level_pipe.values())
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, (max_level + 2) * 3.6), facecolor='white')

        node_text_map_orig = {n: G_orig.nodes[n]['text'] for n in G_orig.nodes()}

        draw_graph_incremental(ax1, G_orig, pos_orig, level_orig, node_text_map_orig, None, False, global_x_limits, global_y_limits, set(G_orig.nodes()))
        ax1.set_title("1. Grafo de Dependência Original", fontsize=15, fontweight='bold', color='#0f172a', pad=15)

        draw_graph_incremental(ax2, G_pipe, pos_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe, True, global_x_limits, global_y_limits, visible_pipe_nodes)
        ax2.set_title("2. Grafo com Software Pipelining (Evolução Nó por Nó)", fontsize=15, fontweight='bold', color='#0f172a', pad=15)

        fig.subplots_adjust(left=0.02, right=0.93, top=0.90, bottom=0.05, wspace=0.15)
        plt.show()

def on_graph_prev_clicked(b):
    global current_graph_idx
    if current_graph_idx > 0:
        current_graph_idx -= 1
        render_current_graph_step()

def on_graph_next_clicked(b):
    global current_graph_idx
    if 'program_instructions' in globals() and program_instructions:
        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)

        level_orig = compute_layered_positions(G_orig)
        _, _, _, _, nodes_order = build_pipelined_with_moves_ordered(G_orig, level_orig)
        if current_graph_idx < len(nodes_order) - 1:
            current_graph_idx += 1
            render_current_graph_step()

def on_graph_final_clicked(b):
    global current_graph_idx
    if 'program_instructions' in globals() and program_instructions:
        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)

        level_orig = compute_layered_positions(G_orig)
        _, _, _, _, nodes_order = build_pipelined_with_moves_ordered(G_orig, level_orig)
        current_graph_idx = len(nodes_order) - 1
        render_current_graph_step()

btn_graph_prev.on_click(on_graph_prev_clicked)
btn_graph_next.on_click(on_graph_next_clicked)
btn_graph_final.on_click(on_graph_final_clicked)

controls_box = widgets.HBox([btn_graph_prev, btn_graph_next, btn_graph_final, lbl_graph_step], layout=widgets.Layout(margin='0 0 10px 0'))
display(controls_box, pipe_compare_output)

if 'program_instructions' in globals() and program_instructions:
    current_graph_idx = 0
    render_current_graph_step()

Output(layout=Layout(margin='14px 0 0 0'))

### Gera código com soft. pipeline e original e compara a saída de ambos

#### Usuário propões Preâmbulo, Loop e Epílogo

In [101]:
# ---------------- Widgets da Ferramenta de Desafio e Validação ----------------
challenge_title = widgets.HTML("""
    <div style="background: linear-gradient(135deg, #2e1065 0%, #3b0764 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;">🏆 Sandbox de Validação: Teste seu Próprio Pipeline</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha sua escala de código. O validador agora usa o simulador oficial da Célula 3 para julgar a equivalência matemática.</p>
    </div>
""")

user_preamble_input = widgets.Textarea(
    value='# Digite seu preambulo aqui\n',
    placeholder='Instruções do Preâmbulo...',
    description='Preâmbulo:',
    layout=widgets.Layout(width='98%', height='100px', margin='0 0 10px 0')
)

user_kernel_input = widgets.Textarea(
    value='# Digite seu miolo do loop aqui\n',
    placeholder='Instruções do Loop...',
    description='Loop:',
    layout=widgets.Layout(width='98%', height='100px', margin='0 0 10px 0')
)

user_epilogue_input = widgets.Textarea(
    value='# Digite seu epilogo aqui\n',
    placeholder='Instruções do Epílogo...',
    description='Epílogo:',
    layout=widgets.Layout(width='98%', height='100px', margin='0 0 15px 0')
)

btn_validate_challenge = widgets.Button(
    description='Validar Equivalência Semântica', icon='check-double',
    button_style='primary', layout=widgets.Layout(width='320px', height='42px')
)

challenge_output = widgets.Output(layout=widgets.Layout(margin='16px 0 0 0'))

def on_validate_challenge_clicked(b):
    with challenge_output:
        clear_output()

        # Verifica se o programa original existe na Célula 2
        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Código base não encontrado. Certifique-se de carregar as instruções na Célula 2.")
            return

        # 1. GABARITO OFICIAL: Executa o programa original usando o PipelineValidator oficial
        # Ele já sabe quantas iterações rodar com base no machine_config.forced_loops
        validator_oficial = PipelineValidator(program_instructions, machine_config)
        try:
            mem_gabarito, _, total_iterations = validator_oficial.compute_reference()
        except Exception as e:
            print(f"❌ Erro ao rodar o programa de referência: {e}")
            return

        # 2. CONSTRUÇÃO DO FLUXO DO USUÁRIO
        # Limpa comentários e linhas vazias de cada bloco do widget
        def clean_block(text):
            return [line.split('#')[0].strip() for line in text.split('\n') if line.split('#')[0].strip()]

        pre_lines = clean_block(user_preamble_input.value)
        ker_lines = clean_block(user_kernel_input.value)
        epi_lines = clean_block(user_epilogue_input.value)

        # Monta o programa linear do usuário repetindo o miolo baseado nas iterações oficiais da máquina
        user_program_text = "\n".join(pre_lines + (ker_lines * total_iterations) + epi_lines)

        # 3. PARSING E SIMULAÇÃO VIA CLASSES OFICIAIS
        try:
            # Usa o parse_program genérico da Célula 1!
            user_parsed_instructions = parse_program(user_program_text)

            # Instancia o simulador real consumindo a configuração ativa da máquina
            simulador_sandbox = ISASimulator(machine_config)

            # Executa o código linear proposto pelo usuário
            _, _, mem_usuario, _ = simulador_sandbox.run_full_program(user_parsed_instructions)
        except Exception as e:
            print(f"❌ Erro de parsing ou execução no seu código proposto: {e}")
            return

        # 4. RENDERIZAÇÃO DA TABELA DE COMPARAÇÃO DE MEMÓRIA
        th_style = "padding: 10px; background-color: #1e293b; color: #f1f5f9; font-size: 13px;"
        td_style = "padding: 8px 10px; border-bottom: 1px solid #1e293b; font-family: monospace; font-size: 13px;"

        table_rows = ""
        mismatches_count = 0
        n_show = min(30, machine_config.mem_size)

        for i in range(n_show):
            v_init = float(i)
            v_gab = mem_gabarito[i]
            v_usr = mem_usuario[i]

            is_modified = (v_gab != v_init)
            row_bg = "background-color: #020617;" if is_modified else ""

            if v_gab == v_usr:
                status = "✔ Correto"
                status_color = "#4ade80"
            else:
                status = "❌ Erro"
                status_color = "#f87171"
                mismatches_count += 1

            table_rows += f"""
            <tr style="{row_bg}">
                <td style="{td_style} color: #94a3b8; font-weight: bold;">Índice [{i}]</td>
                <td style="{td_style} color: #64748b;">{v_init:.1f}</td>
                <td style="{td_style} color: #38bdf8;">{v_gab:.1f}</td>
                <td style="{td_style} color: #e2e8f0;">{v_usr:.1f}</td>
                <td style="{td_style} color: {status_color}; font-weight: bold;">{status}</td>
            </tr>
            """

        if mismatches_count == 0:
            veredicto_html = f"""
            <div style="margin-top: 16px; padding: 16px; background-color: #064e3b; border-left: 4px solid #10b981; color: #34d399; border-radius: 6px; font-family: sans-serif; font-size: 14px;">
                🎉 <b>Veredicto: PIPELINE APROVADO!</b> Seu arranjo de instruções respeitou todas as dependências temporais. O resultado final gerado na simulação é perfeitamente equivalente ao sequencial para {total_iterations} iterações.
            </div>
            """
        else:
            veredicto_html = f"""
            <div style="margin-top: 16px; padding: 16px; background-color: #4c0519; border-left: 4px solid #f43f5e; color: #f43f5e; border-radius: 6px; font-family: sans-serif; font-size: 14px;">
                ❌ <b>Veredicto: FALHA DE EQUIVALÊNCIA!</b> O simulador detectou {mismatches_count} divergências de dados na memória. Revise a ordem dos seus estágios ou a lógica de encaminhamento dos MOVs.
            </div>
            """

        html_view = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; font-weight: 600; font-size: 18px; margin-bottom: 12px;">📊 Verificação de Impacto no Vetor de Dados (Configuração Ativa)</h4>
            <div style="background-color: #0f172a; padding: 12px; border-radius: 8px; max-height: 400px; overflow-y: auto;">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: sans-serif;">
                    <thead>
                        <tr>
                            <th style="{th_style}">Vetor</th>
                            <th style="{th_style}">Estado Inicial</th>
                            <th style="{th_style}">Gabarito Sequencial</th>
                            <th style="{th_style}">Seu Pipeline</th>
                            <th style="{th_style}">Resultado</th>
                        </tr>
                    </thead>
                    <tbody>{table_rows}</tbody>
                </table>
            </div>
            {veredicto_html}
        """, layout=widgets.Layout(width='100%'))

        display(html_view)

btn_validate_challenge.on_click(on_validate_challenge_clicked)

user_preamble_input.add_class('widget-textarea')
user_kernel_input.add_class('widget-textarea')
user_epilogue_input.add_class('widget-textarea')

form_box = widgets.VBox([
    challenge_title,
    widgets.VBox([
        user_preamble_input,
        user_kernel_input,
        user_epilogue_input,
        widgets.HBox([btn_validate_challenge], layout=widgets.Layout(align_items='center'))
    ], layout=widgets.Layout(padding='20px', border='1px solid #dcdde1', border_radius='0 0 12px 14px', background_color='#fafbfc'))
])

display(form_box, challenge_output)

Output(layout=Layout(margin='16px 0 0 0'))

#### Gabarito

In [118]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import defaultdict
import re
import networkx as nx

# ---------------- Widgets da Célula de Texto do Pipeline ----------------
asm_compare_button = widgets.Button(
    description='Gerar Código e Simular Matrizes', icon='play',
    button_style='success', layout=widgets.Layout(width='380px', height='40px')
)
asm_compare_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

# ⚙️ FUNÇÃO DE SUPORTE LOCAL
def extract_dest_reg_local(text):
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def build_pipelined_with_moves(G_orig, level_orig):
    """Gera o grafo de pipeline completo mapeando registradores por nível"""
    all_regs = []
    for node in G_orig.nodes():
        all_regs.extend(re.findall(r'f\d+', G_orig.nodes[node]['text'].lower()))
    reg_indices = [int(r[1:]) for r in all_regs if r.startswith('f')]
    max_reg_idx = max(reg_indices) if reg_indices else 0
    global_next_reg = max_reg_idx + 1

    G_pipe = nx.DiGraph()
    level_pipe, node_text_map, dest_changed_map = {}, {}, {}
    levels_to_nodes = defaultdict(list)
    for node, lvl in level_orig.items():
        levels_to_nodes[lvl].append(node)

    reg_name_at_level = defaultdict(dict)
    mov_nodes, move_node_counter, seen_destinations = {}, 1000, set()

    prod_consumers = defaultdict(list)
    for u, v, data in G_orig.edges(data=True):
        for r in data['regs']:
            prod_consumers[(u, r.lower())].append(v)

    for u in G_orig.nodes():
        orig_text = G_orig.nodes[u]['text'].lower()
        d = extract_dest_reg_local(orig_text)
        if d: reg_name_at_level[(u, d)][level_orig[u]] = d

    for lvl in sorted(levels_to_nodes.keys()):
        for (p, r), consumers in list(prod_consumers.items()):
            L_p = level_orig[p]
            L_end = max(level_orig[c] for c in consumers)
            if L_p < lvl <= L_end:
                if lvl < L_end:
                    mov_id = move_node_counter
                    move_node_counter += 1
                    r_prev = reg_name_at_level[(p, r)][lvl - 1]
                    r_new = f"f{global_next_reg}"
                    global_next_reg += 1

                    node_text_map[mov_id] = f"mov {r_new}, {r_prev}"
                    level_pipe[mov_id] = lvl
                    dest_changed_map[mov_id] = True
                    G_pipe.add_node(mov_id)
                    mov_nodes[(p, r, lvl)] = mov_id
                    reg_name_at_level[(p, r)][lvl] = r_new
                    parent = p if lvl - 1 == L_p else mov_nodes[(p, r, lvl - 1)]
                    G_pipe.add_edge(parent, mov_id, regs={r_prev})
                else:
                    reg_name_at_level[(p, r)][lvl] = reg_name_at_level[(p, r)][lvl - 1]

        for u in levels_to_nodes[lvl]:
            orig_text = G_orig.nodes[u]['text'].lower()
            src_mappings = {}
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    src_mappings[r.lower()] = reg_name_at_level[(p_node, r.lower())][lvl - 1]

            orig_dest = extract_dest_reg_local(orig_text)
            new_dest = orig_dest
            dest_changed = False
            if orig_dest:
                if orig_dest in seen_destinations:
                    new_dest = f"f{global_next_reg}"
                    global_next_reg += 1
                    dest_changed = True
                seen_destinations.add(orig_dest)

            parts = orig_text.split(None, 1)
            if len(parts) == 2:
                opcode, rest = parts; subparts = rest.split(',')
                is_store = opcode in ['sd', 'sw', 'sb', 'sh']
                new_subparts = []
                for idx, sub in enumerate(subparts):
                    sub_clean = sub.strip()
                    if idx == 0 and not is_store and orig_dest is not None:
                        new_subparts.append(new_dest)
                    else:
                        updated_sub = sub_clean
                        for orig_src, mapped_name in src_mappings.items():
                            updated_sub = re.sub(r'\b' + re.escape(orig_src) + r'\b', mapped_name, updated_sub)
                        new_subparts.append(updated_sub)
                updated_text = f"{opcode} {', '.join(new_subparts)}"
            else:
                updated_text = orig_text

            node_text_map[u] = updated_text
            level_pipe[u] = lvl
            dest_changed_map[u] = dest_changed
            G_pipe.add_node(u)

            if orig_dest: reg_name_at_level[(u, orig_dest)][lvl] = new_dest
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    parent_in_pipe = p_node if lvl - 1 == level_orig[p_node] else mov_nodes[(p_node, r.lower(), lvl - 1)]
                    G_pipe.add_edge(parent_in_pipe, u, regs={reg_name_at_level[(p_node, r.lower())][lvl - 1]})

    return G_pipe, level_pipe, node_text_map, dest_changed_map

def generate_asm_text_blocks(level_pipe, node_text_map_pipe, original_instructions):
    """Gera o código Assembly real e executável aplicando a matemática correta de offsets"""
    stages_pipe = defaultdict(list)
    for node in sorted(node_text_map_pipe.keys()):
        lvl = level_pipe[node]
        stages_pipe[lvl].append(node_text_map_pipe[node])

    max_lvl = max(level_pipe.values()) if level_pipe else 0
    strides = ISASimulator.detect_induction_strides(original_instructions)
    stride = abs(list(strides.values())[0]) if strides else 8

    control_instructions = [
        inst['raw'].lower() for inst in original_instructions
        if inst['category'] in ('branch', 'jump') or
        (inst['category'] == 'i_type' and inst['operands'][0] == inst['operands'][1])
    ]

    def adjust_offsets(text, s, mode, c=0):
        def repl(m):
            offset = int(m.group(1))
            base = m.group(2)
            if base.startswith('r'):
                if mode == 'pre':
                    return f"{offset + (c - s) * stride}({base})"
                return f"{offset + (max_lvl - s) * stride}({base})"
            return m.group(0)
        return re.sub(r'(-?\d+)\((\w+)\)', repl, text)

    # 1. PREÂMBULO (Offsets progressivos calculados por ciclo)
    pre_lines = ["# ==========================================",
                 "#              1. PREÂMBULO                  ",
                 "# =========================================="]
    for c in range(max_lvl):
        pre_lines.append(f"\n# --- Ciclo {c} do Preâmbulo ---")
        for s in range(c, -1, -1):
            for inst in stages_pipe[s]:
                inst_adj = adjust_offsets(inst, s, 'pre', c)
                pre_lines.append(f"    {inst_adj:<25} # [Iteração i+{c-s}]")

    # 2. KERNEL (Rótulo Inline + Espaçamento correto entre iterações)
    kernel_lines = ["# ==========================================",
                    "#       2. LOOP PRINCIPAL (KERNEL)          ",
                    "# =========================================="]
    is_first = True
    for s in sorted(stages_pipe.keys(), reverse=True):
        iter_str = "i" if s == 0 else f"i-{s}"
        for inst in stages_pipe[s]:
            inst_adj = adjust_offsets(inst, s, 'kernel')
            if is_first:
                kernel_lines.append(f"LOOP_PIPE: {inst_adj:<25} # [Iteração {iter_str}]")
                is_first = False
            else:
                kernel_lines.append(f"    {inst_adj:<25} # [Iteração {iter_str}]")
    for ctrl in control_instructions:
        if 'bne' in ctrl or 'beq' in ctrl:
            ctrl_renamed = re.sub(r'\b\w+$', 'LOOP_PIPE', ctrl)
            kernel_lines.append(f"    {ctrl_renamed}")
        else:
            kernel_lines.append(f"    {ctrl}")

    # 3. EPÍLOGO (Sincronizado com o avanço do laço)
    epi_lines = ["# ==========================================",
                 "#               3. EPÍLOGO                  ",
                 "# =========================================="]
    for c in range(1, max_lvl + 1):
        epi_lines.append(f"\n# --- Ciclo {c} do Epílogo ---")
        for s in range(max_lvl, c - 1, -1):
            for inst in stages_pipe[s]:
                inst_adj = adjust_offsets(inst, s, 'epi')
                iter_offset = s - c
                iter_str = "fim" if iter_offset == 0 else f"fim-{iter_offset}"
                epi_lines.append(f"    {inst_adj:<25} # [Iteração {iter_str}]")
        for ctrl in control_instructions:
            if 'bne' not in ctrl and 'beq' not in ctrl:
                epi_lines.append(f"    {ctrl}")

    return "\n".join(pre_lines), "\n".join(kernel_lines), "\n".join(epi_lines)

def on_asm_compare_clicked(b):
    with asm_compare_output:
        clear_output()

        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Nenhuma instrução encontrada no programa. Adicione instruções na primeira célula.")
            return

        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)

        if G_orig.number_of_nodes() == 0:
            print("ℹ️ Nenhuma dependência de registradores para processar.")
            return

        level_orig = compute_layered_positions(G_orig)
        G_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe = build_pipelined_with_moves(G_orig, level_orig)

        # GERA O CÓDIGO REAL COM OS OFFSETS CORRIGIDOS MATEMATICAMENTE
        pre_text, kernel_text, epi_text = generate_asm_text_blocks(level_pipe, node_text_map_pipe, program_instructions)

        orig_insts_list = [G_orig.nodes[n]['text'].lower() for n in sorted(G_orig.nodes())]
        orig_text = "LOOP_ORIGINAL:\n" + "\n".join(f"    {inst}" for inst in orig_insts_list)

        # ALIMENTA O SEU VALIDADOR DA CÉLULA 3 DIRETAMENTE COM OS NÓS DO GRAFO
        generated_pipeline_nodes = [{'text': node_text_map_pipe[n], 'level': level_pipe[n]} for n in G_pipe.nodes()]

        validator = PipelineValidator(program_instructions, machine_config)
        result = validator.validate(generated_pipeline_nodes)

        if result.get("error"):
            print(f"❌ {result['error']}")
            return

        # RENDERIZAÇÃO DA INTERFACE VISUAL
        code_style = (
            "background-color: #0f172a; color: #38bdf8; padding: 16px; "
            "border-radius: 10px; font-family: 'Consolas', monospace; "
            "font-size: 14px; line-height: 1.6; overflow-x: auto; white-space: pre;"
        )

        html_original = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; margin-bottom: 8px; font-weight: 600; font-size: 20px;">🔄 Código Original (Sequencial)</h4>
            <div style="{code_style} color: #e2e8f0;">{orig_text}</div>
        """, layout=widgets.Layout(width='35%'))

        html_pipelined = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; margin-bottom: 8px; font-weight: 600; font-size: 20px;">🚀 Código Otimizado Gerado com Offsets Reais</h4>
            <div style="{code_style} margin-bottom: 12px; border-left: 4px solid #f59e0b; color: #fbbf24;">{pre_text}</div>
            <div style="{code_style} margin-bottom: 12px; border-left: 4px solid #10b981; color: #34d399;">{kernel_text}</div>
            <div style="{code_style} border-left: 4px solid #ef4444; color: #f87171;">{epi_text}</div>
        """, layout=widgets.Layout(width='62%'))

        layout_codigo = widgets.HBox([html_original, html_pipelined], layout=widgets.Layout(gap='20px', width='100%'))

        th_style = "padding: 10px; background-color: #1e293b; color: #f1f5f9; position: sticky; top: 0; font-size: 13px;"
        td_style = "padding: 8px 10px; border-bottom: 1px solid #1e293b; font-family: monospace; font-size: 13px;"

        table_rows = ""
        mem_gabarito = result["mem_gabarito"]
        mem_pipe = result["mem_pipe"]
        n_show = min(30, machine_config.mem_size)

        for i in range(n_show):
            v_init = float(i)
            v_mem = mem_gabarito[i]
            v_l = mem_pipe[i]

            is_modified = (v_mem != v_init)
            row_bg = "background-color: #111827;" if is_modified else ""
            idx_style = "color: #38bdf8; font-weight: bold;" if is_modified else "color: #94a3b8;"
            status = "✔ Perfeito" if v_mem == v_l else "❌ Mismatch"
            status_color = "#4ade80" if v_mem == v_l else "#f87171"

            table_rows += f"""
            <tr style="{row_bg} border-bottom: 1px solid #1e293b;">
                <td style="{td_style} {idx_style}">Índice [{i}]</td>
                <td style="{td_style} color: #64748b;">{v_init}</td>
                <td style="{td_style} color: #38bdf8;">{v_mem:.1f}</td>
                <td style="{td_style} color: #34d399;">{v_l:.1f}</td>
                <td style="{td_style} color: {status_color}; font-weight: bold;">{status}</td>
            </tr>
            """

        status_veredicto = "✔ <b>Sucesso na Homologação:</b> O estado final bateu 100% idêntico!" if result["success"] else "❌ <b>Falha na Homologação:</b> Ocorreu divergência matemática."
        bg_veredicto = "#14532d" if result["success"] else "#4c0519"
        color_veredicto = "#4ade80" if result["success"] else "#f43f5e"

        html_tables = widgets.HTML(f"""
            <hr style="border: 0; border-top: 1px solid #334155; margin: 24px 0;">
            <h4 style="color: #f1f5f9; font-family: sans-serif; font-weight: 600; font-size: 20px; margin-bottom: 4px;">📊 Comparação Semântica Unificada (Célula 3)</h4>
            <div style="background-color: #0f172a; padding: 12px; border-radius: 8px; max-height: 450px; overflow-y: auto; width: 100%;">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: sans-serif;">
                    <thead>
                        <tr>
                            <th style="{th_style}">Posição do Vetor</th>
                            <th style="{th_style}">Valor Inicial</th>
                            <th style="{th_style}">Loop Original (Gabarito)</th>
                            <th style="{th_style}">Código Otimizado Automático</th>
                            <th style="{th_style}">Status de Validação</th>
                        </tr>
                    </thead>
                    <tbody>{table_rows}</tbody>
                </table>
            </div>
            <div style="margin-top: 16px; padding: 14px; background-color: {bg_veredicto}; color: {color_veredicto}; border-radius: 6px; font-family: sans-serif; font-size: 14px; font-weight: 500;">
                {status_veredicto}
            </div>
        """, layout=widgets.Layout(width='100%'))

        display(widgets.VBox([layout_codigo, html_tables]))

asm_compare_button.on_click(on_asm_compare_clicked)
asm_compare_output.add_class('output-box')
display(asm_compare_button, asm_compare_output)

Button(button_style='success', description='Gerar Código e Simular Matrizes', icon='play', layout=Layout(heigh…

Output(layout=Layout(margin='14px 0 0 0'), _dom_classes=('output-box',))

### Ciclos com Escalonamento Dinâmico + Comp. de Cálculo de CPI

In [134]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import defaultdict
import re
import networkx as nx
from tabulate import tabulate
import copy

# =========================================================================
# 🧠 INTERACTIVE TOMASULO ENGINE (COM MOTOR HISTÓRICO E SUPORTE IMEDIATO)
# =========================================================================
class TomasuloEngineInteractive:
    def __init__(self, raw_lines):
        self.clk = 1
        self.stall = (None, None)
        self.done_branch = False
        self.stop = False
        self.forced_stop = False
        self.bne_predicted_inst = None
        self.prev_fwd = {}
        self.pipeline = {'fetch': None, 'decode': None}
        self.issue_ticket = 0
        self.stations_to_write = {}
        self.stations_to_reset = []

        # Latências de Hardware
        self.cicles = {'l': 2, 'a': 3, 'm': 5, 'i': 1, 'v': 1}  # 'v' = mov
        self.mov_counter = 0

        # Estações de Reserva
        self.stations = {}
        for op, cnt in [('l', 3), ('m', 2), ('a', 2), ('i', 1)]:
            for i in range(cnt):
                self.stations[f"{op}{i+1}"] = {
                    'inst': None, 'regs': [None]*3, 'end': None,
                    'fwd': None, 'write': None, 'branch': False, 'ticket': None
                }
        self.stage = {s: {'execute': None} for s in self.stations}

        # Banco de Registradores e Memória
        self.registers = {f'r{i}': i * 4 for i in range(1, 5)}
        self.registers.update({f'f{i}': float(i) for i in range(1, 13)})
        self.memory = [float(x) for x in range(32)]

        # Processamento e normalização da fila de instruções
        self.instructions = []
        seen_counts = defaultdict(int)
        for line in raw_lines:
            clean = line.split('#')[0].strip().lower()
            if not clean: continue
            if ":" in clean: clean = clean.split(":", 1)[1].strip()

            # Casamento de opcodes com a ISA do Tomasulo
            clean = re.sub(r'\bmult\b', 'mulf', clean)
            clean = re.sub(r'\badd\b(?=\s+f)', 'addf', clean)

            seen_counts[clean] += 1
            unique_name = f"{clean}   #{seen_counts[clean]}" if seen_counts[clean] > 1 else clean
            self.instructions.append(unique_name)

        self.branch_instructions = self.instructions.copy()
        self.x_headers = ["fetch", "decode", "execute", "write", "fetch_1", "decode_1", "execute_1", "write_1"]
        self.y_headers = self.instructions.copy()
        self.table_data = {y: {h: "" for header in self.x_headers for h in [header]} for y in self.y_headers}

    def parse_op(self, reg):
        try: return int(reg)
        except ValueError:
            val = self.registers.get(reg, 0.0)
            if isinstance(val, tuple): return val[0]
            return val

    def write_same_clk(self, this_id, delay):
        write = self.stations[this_id]['write']
        for id in [s for s in self.stations if s != this_id]:
            if this_id[0] == 'i' or id[0] == 'i': continue
            if (write and self.stations[id]['write']) and self.stations[id]['end'] == delay: return True
        return False

    def check_dependencies(self, id, op, regs):
        def get_val(r):
            v = self.registers.get(r, 0.0)
            return v[0] if isinstance(v, tuple) else v

        reg_0_val = get_val(regs[0])
        reg_1_val = None if op in ['sd','ld'] else get_val(regs[1])
        reg_2_val = self.parse_op(regs[2])
        delay = self.clk

        if reg_0_val is not None and reg_1_val is not None and reg_2_val is not None:
            if isinstance(reg_1_val, str) and self.stations[reg_1_val]['end']:
                delay = max(delay, self.stations[reg_1_val]['end'])
            if isinstance(reg_2_val, str) and self.stations[reg_2_val]['end']:
                delay = max(delay, self.stations[reg_2_val]['end'])

        self.stations[id]['write'] = False if op == 'sd' else True
        if op == 'sd' and isinstance(reg_0_val, str) and self.stations[reg_0_val]['end']:
            delay = max(delay, self.stations[reg_0_val]['end'])

        if op in ['add', 'addi', 'sub', 'subi', 'bne']: delay += self.cicles['i']
        else: delay += self.cicles['l' if op == 'sd' else (op[0] if op[0] in self.cicles else 'i')]

        while self.write_same_clk(id, delay): delay += 1

        self.stations[id]['end'] = delay
        self.stations[id]['regs'][0] = reg_0_val if op == 'sd' else regs[0]
        self.stations[id]['regs'][1] = int(regs[1]) if op in ['sd','ld'] else reg_1_val
        self.stations[id]['regs'][2] = reg_2_val

        if op != 'sd':
            old_val = self.registers.get(regs[0], 0.0)
            if isinstance(old_val, tuple): old_val = old_val[1]
            self.registers[regs[0]] = (id, old_val, self.issue_ticket)
            self.stations[id]['ticket'] = self.issue_ticket
            self.issue_ticket += 1

    def check_dependencies_mov(self, id, dest, src):
        def get_val(r):
            v = self.registers.get(r, 0.0)
            return v[0] if isinstance(v, tuple) else v

        src_val = get_val(src)
        delay = self.clk

        if isinstance(src_val, str) and src_val in self.stations and self.stations[src_val]['end']:
            delay = max(delay, self.stations[src_val]['end'])

        self.stations[id]['write'] = True
        delay += self.cicles['v']

        while self.write_same_clk(id, delay): delay += 1

        self.stations[id]['end'] = delay
        self.stations[id]['regs'][0] = dest
        self.stations[id]['regs'][1] = src_val
        self.stations[id]['regs'][2] = None

        old_val = self.registers.get(dest, 0.0)
        if isinstance(old_val, tuple): old_val = old_val[1]
        self.registers[dest] = (id, old_val, self.issue_ticket)
        self.stations[id]['ticket'] = self.issue_ticket
        self.issue_ticket += 1

    def get_station(self, op, clk):
        target_op = "i" if op in ["add", "addi", "sub", "subi", "bne"] else ('l' if op == 'sd' else op[0])
        for id in [s for s in self.stations if s.startswith(target_op)]:
            if self.stations[id]['end'] is None or self.stations[id]['end'] < clk: return id
        return None

    def next_available_station(self, op):
        free_clk = None
        target_op = "i" if op in ["add", "addi", "sub", "subi"] else ('l' if op == 'sd' else op[0])
        for id in [s for s in self.stations if s.startswith(target_op)]:
            if self.stations[id]['end']:
                free_clk = self.stations[id]['end'] if free_clk is None else min(free_clk, self.stations[id]['end'])
        return free_clk

    def reset_station(self, id):
        self.stations[id] = {'inst': None, 'regs': [None]*3, 'end': None, 'fwd': None, 'write': None, 'branch': False, 'ticket': None}

    def is_done(self):
        if self.forced_stop: return True
        return len(self.instructions) == 0 and self.pipeline['decode'] is None and all(self.stations[s]['end'] is None or self.stations[s]['end'] < self.clk for s in self.stations) and len(self.stations_to_write) == 0

    def step_cycle(self):
        if self.is_done(): return

        # WRITE
        for id in list(self.stations_to_write.keys()):
            if self.stations_to_write[id]['end'] is None or self.stations_to_write[id]['end'] + 1 != self.clk: continue
            tagw = 'write_1' if self.stations_to_write[id]['branch'] else 'write'
            inst = self.stations_to_write[id]['inst']

            target_key = inst
            for k in self.table_data:
                if k.startswith(inst): target_key = k; break

            current_content = self.table_data[target_key][tagw]
            if not inst.startswith('sd') and not inst.startswith('bne'):
                self.table_data[target_key][tagw] = f"{self.clk} {current_content.split()[-1] if current_content else id}"

            if self.stations_to_write[id]['write']:
                reg_dest = self.stations_to_write[id]['regs'][0]
                val_fwd = self.stations_to_write[id]['fwd']
                estado_atual = self.registers.get(reg_dest, 0.0)
                if isinstance(estado_atual, tuple):
                    if estado_atual[0] == id and len(estado_atual) == 3 and estado_atual[2] == self.stations_to_write[id]['ticket']:
                        self.registers[reg_dest] = val_fwd
                    else: self.registers[reg_dest] = (estado_atual[0], val_fwd, estado_atual[2])
                else: self.registers[reg_dest] = val_fwd
            elif self.stations_to_write[id]['inst'] and self.stations_to_write[id]['inst'].startswith('sd'):
                val_to_write = self.stations_to_write[id]['regs'][0]
                endereco_memoria = min(max(0, self.stations_to_write[id]['regs'][1] + self.stations_to_write[id]['regs'][2]), len(self.memory) - 1)
                self.memory[endereco_memoria] = val_to_write

        self.prev_fwd = {sid: sw['fwd'] for sid, sw in self.stations_to_write.items() if sw['fwd'] is not None}
        self.stations_to_write = {}

        # FORWARD
        for id in self.stations:
            if self.stations[id]['end'] is None or self.stations[id]['end'] < self.clk: continue
            tagf = 'execute_1' if self.stations[id]['branch'] else 'execute'
            regs = self.stations[id]['regs']

            if self.stations[id]['end'] == self.clk:
                inst = self.stations[id]['inst']
                if inst.startswith('bne'):
                    reg1_val, reg2_val = self.stations[id]['regs'][1], self.stations[id]['regs'][2]
                    if not self.stop:
                        if reg1_val != reg2_val:
                            self.instructions = self.branch_instructions.copy()
                            self.done_branch = True
                        else: self.stop = True
                    self.stations_to_write[id] = self.stations[id].copy()
                    self.stations_to_reset.append(id)
                    continue

                target_key = inst
                for k in self.table_data:
                    if k.startswith(inst): target_key = k; break

                if not id.startswith('i'): self.table_data[target_key][tagf] += f'{self.clk}'

                fwd = None
                if self.stations[id]['write']:
                    r1 = self.stations[regs[1]]['fwd'] if (isinstance(regs[1], str) and regs[1] in self.stations and self.stations[regs[1]]['fwd'] is not None) else self.prev_fwd.get(regs[1], regs[1])
                    r2 = self.stations[regs[2]]['fwd'] if (isinstance(regs[2], str) and regs[2] in self.stations and self.stations[regs[2]]['fwd'] is not None) else self.prev_fwd.get(regs[2], regs[2])
                    self.stations[id]['regs'][1], self.stations[id]['regs'][2] = r1, r2

                    if id[0] == 'l': fwd = self.memory[min(max(0, r1 + r2), len(self.memory) - 1)]
                    elif id[0] == 'm': fwd = r1 * r2
                    elif id[0] == 'i': fwd = r1 - r2 if self.stations[id]['inst'].split()[0] in ['sub', 'subi'] else r1 + r2
                    elif id[0] == 'v': fwd = r1
                    else: fwd = r1 + r2

                self.stations[id]['fwd'] = fwd
                self.stations_to_write[id] = self.stations[id].copy()
                self.stations_to_reset.append(id)

        # EXECUTE
        for id in self.stations:
            if self.stations[id]['end'] is None or self.stations[id]['end'] < self.clk: continue
            tage = 'execute_1' if self.stations[id]['branch'] else 'execute'
            tagw = 'write_1' if self.stations[id]['branch'] else 'write'
            inst = self.stations[id]['inst']

            if self.clk == (self.stations[id]['end'] - self.cicles.get(id[0], 1) + 1 if not id.startswith('i') else self.stations[id]['end']):
                target_key = inst
                for k in self.table_data:
                    if k.startswith(inst): target_key = k; break
                self.table_data[target_key][tage] = f'{self.clk}'
                self.table_data[target_key][tagw] = f'{id}'

            if not id.startswith('i') and self.stations[id]['end'] - self.cicles[id[0]] + 1 == self.clk:
                target_key = inst
                for k in self.table_data:
                    if k.startswith(inst): target_key = k; break
                if self.table_data[target_key][tage] == f'{self.clk}': self.table_data[target_key][tage] += '-'
                else: self.table_data[target_key][tage] += f'..{self.clk}-'

            for r_idx in [1, 2]:
                r_val = self.stations[id]['regs'][r_idx]
                if isinstance(r_val, str) and r_val in self.stations:
                    self.stations[id]['regs'][r_idx] = self.stations[r_val]['fwd'] if self.stations[r_val]['fwd'] is not None else self.prev_fwd.get(r_val, r_val)

        for id in self.stations_to_reset: self.reset_station(id)
        self.stations_to_reset = []

        # DECODE
        tagd_d = 'decode_1' if self.done_branch else 'decode'
        if self.pipeline['decode']:
            inst = self.pipeline['decode']
            clean_inst = inst.split('   #')[0].strip()

            if ":" in clean_inst:
                clean_inst = clean_inst.split(":", 1)[1].strip()

            if clean_inst.startswith('bne') and not self.stop:
                regs_bne = clean_inst.replace('bne', '').strip().split(',')
                station_id = self.get_station('bne', self.clk)
                if station_id:
                    if not self.done_branch:
                        self.instructions = self.branch_instructions.copy()
                        self.done_branch = True
                    else: self.stop = True
                    self.stations[station_id]['inst'] = clean_inst
                    self.stations[station_id]['write'] = False
                    self.stations[station_id]['regs'] = [None, self.parse_op(regs_bne[0].strip()), self.parse_op(regs_bne[1].strip())]
                    self.stations[station_id]['end'] = self.clk + self.cicles['i']
                    self.table_data[inst][tagd_d] = str(self.clk)
                    self.pipeline['decode'] = None
                    self.stall = (None, None)
            elif clean_inst.startswith('mov'):
                regs_mov = clean_inst.replace('mov', '').strip().split(',')
                dest_reg, src_reg = regs_mov[0].strip(), regs_mov[1].strip()

                self.mov_counter += 1
                station_id = f"v{self.mov_counter}"
                self.stations[station_id] = {
                    'inst': None, 'regs': [None] * 3, 'end': None,
                    'fwd': None, 'write': None, 'branch': False, 'ticket': None
                }
                if self.done_branch: self.stations[station_id]['branch'] = True
                self.stations[station_id]['inst'] = clean_inst
                self.check_dependencies_mov(station_id, dest_reg, src_reg)

                if self.table_data[inst][tagd_d] and self.table_data[inst][tagd_d].endswith('-'): self.table_data[inst][tagd_d] += str(self.clk)
                else: self.table_data[inst][tagd_d] = f"{self.table_data[inst][tagd_d]}, {self.clk}" if self.table_data[inst][tagd_d] else str(self.clk)
                self.pipeline['decode'] = None
                self.stall = (None, None)
            else:
                splitted = clean_inst.split(maxsplit=1)
                op = splitted[0]
                if clean_inst.startswith('ld') or clean_inst.startswith('sd'):
                    reg1, rest = splitted[1].replace(" ", "").split(',', 1)
                    offset, reg2 = rest.split('(', 1)
                    regs = [reg1, offset, reg2.rstrip(')')]
                else: regs = splitted[1].replace(" ", "").split(',')

                station_id = self.get_station(op, self.clk)
                if station_id:
                    if self.done_branch: self.stations[station_id]['branch'] = True
                    self.stations[station_id]['inst'] = clean_inst
                    self.check_dependencies(station_id, op, regs)
                    if self.table_data[inst][tagd_d] and self.table_data[inst][tagd_d].endswith('-'): self.table_data[inst][tagd_d] += str(self.clk)
                    else: self.table_data[inst][tagd_d] = f"{self.table_data[inst][tagd_d]}, {self.clk}" if self.table_data[inst][tagd_d] else str(self.clk)
                    self.pipeline['decode'] = None
                    self.stall = (None, None)
                else:
                    self.stall = (self.next_available_station(op), clean_inst)
                    if not self.table_data[inst][tagd_d]: self.table_data[inst][tagd_d] = f"{self.clk}-"
                    else: self.table_data[inst][tagd_d] += f", {self.clk}-"

        # FETCH
        tagf = 'fetch_1' if self.done_branch else 'fetch'
        if self.pipeline.get('fetch') is not None and self.pipeline.get('decode') is None:
            inst_in_fetch = self.pipeline['fetch']
            self.pipeline['decode'] = inst_in_fetch
            self.pipeline['fetch'] = None
            if str(self.table_data[inst_in_fetch][tagf]) != str(self.clk): self.table_data[inst_in_fetch][tagf] = f"{self.table_data[inst_in_fetch][tagf]}-{self.clk}"
        elif self.pipeline.get('fetch') is None and len(self.instructions) > 0:
            inst = self.instructions.pop(0)
            self.pipeline['fetch'] = inst
            self.table_data[inst][tagf] = str(self.clk)
            if self.pipeline.get('decode') is None:
                self.pipeline['decode'] = inst; self.pipeline['fetch'] = None

        # 🎯 CRITÉRIO DE PARADA BARREIRA: Interrompe apenas quando TODAS as instruções terminarem na It2
        all_finished = True
        for k, stages in self.table_data.items():
            clean_name = k.split('   #')[0].strip()
            if clean_name.startswith('bne'):
                continue
            if clean_name.startswith('sd'):
                # Como o sd não gera número em Write, validamos o término de sua Execução (It2)
                exe_val = str(stages.get('execute_1', "")).strip()
                if not (exe_val and '-' in exe_val):
                    all_finished = False
                    break
            else:
                # Todas as outras instruções dependem do número de clock preenchido no Write (It2)
                w_val = str(stages.get('write_1', "")).strip()
                if not any(n.isdigit() for n in w_val.split()):
                    all_finished = False
                    break
        if all_finished and len(self.table_data) > 0:
            self.forced_stop = True

    def get_calculated_cpi_info(self):
        """Mede com precisão o CPI do regime permanente (Steady State) isolando o startup do loop"""
        corpo_insts = [k for k in self.table_data.keys() if not k.split('   #')[0].strip().startswith("bne")]

        if not corpo_insts:
            return None, "Aguardando avanço das iterações para cálculo de regime permanente..."

        def get_completion_cycle(stages_dict, clean_name, suffix=""):
            if clean_name.startswith('sd'):
                exe_val = str(stages_dict.get('execute' + suffix, "")).strip()
                nums = [int(n) for n in re.findall(r'\d+', exe_val)]
                return nums[-1] if nums else None
            else:
                w_val = str(stages_dict.get('write' + suffix, "")).strip()
                nums = [int(n) for n in re.findall(r'\d+', w_val)]
                return nums[0] if nums else None

        clks_it1 = []
        clks_it2 = []

        for k in corpo_insts:
            clean_name = k.split('   #')[0].strip()
            c1 = get_completion_cycle(self.table_data[k], clean_name, suffix="")
            c2 = get_completion_cycle(self.table_data[k], clean_name, suffix="_1")
            if c1 is not None: clks_it1.append(c1)
            if c2 is not None: clks_it2.append(c2)

        if len(clks_it1) < len(corpo_insts) or len(clks_it2) < len(corpo_insts):
            return None, "Aguardando a conclusão total (WB/Store) do corpo do loop nas duas iterações..."

        # Captura o exato momento em que o pior caminho de dados terminou em cada laço
        max_clk_it1 = max(clks_it1)
        max_clk_it2 = max(clks_it2)

        num_insts = len(self.table_data)  # Total de instruções por iteração física
        cycle_delta = max_clk_it2 - max_clk_it1
        cpi = cycle_delta / num_insts
        return cpi, f"📈 CPI: {cpi:.2f} | (Cálculo: [Fim_It2={max_clk_it2} - Fim_It1={max_clk_it1}] / {num_insts} instruções)"

# =========================================================================
# ⚙️ GERENCIADOR DE HISTÓRICO DE PASSOS (SNAPSHOT MANAGER)
# =========================================================================
current_sim = None
history_stack = []
reference_original_cpi_str = "Aguardando simulação de referência..."

drop_select_stream = widgets.Dropdown(
    options=[
        ("🏆 Com Soft. Pipeline (Seu Grafo)", 'canvas'),
        ("⚡ Com Soft. Pipeline (Gabarito)", 'gabarito'),
        ("🔄 Loop Original (Sequencial)", 'original')
    ],
    value='canvas',
    description='Código Alvo:',
    layout=widgets.Layout(width='320px')
)

btn_reset_sim = widgets.Button(description='Reiniciar', icon='sync', button_style='warning', layout=widgets.Layout(width='120px', height='40px'))
btn_prev_pc = widgets.Button(description='PC - 1', icon='arrow-left', button_style='danger', layout=widgets.Layout(width='110px', height='40px'))
btn_step_pc = widgets.Button(description='PC + 1', icon='arrow-right', button_style='info', layout=widgets.Layout(width='110px', height='40px'))
btn_final_pc = widgets.Button(description='Ir para o Final', icon='fast-forward', button_style='success', layout=widgets.Layout(width='150px', height='40px'))

dashboard_panel_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

def run_baseline_original_simulation():
    """Gera uma simulação fixa do loop sequencial original para termos o gabarito de comparação direta"""
    global reference_original_cpi_str
    if 'program_instructions' not in globals() or not program_instructions:
        reference_original_cpi_str = "N/A (Carregue o programa primeiro)"
        return
    try:
        orig_stream = [inst['raw'] for inst in program_instructions]
        ref_engine = TomasuloEngineInteractive(orig_stream)
        while not ref_engine.is_done() and ref_engine.clk < 1000:
            ref_engine.step_cycle()
            ref_engine.clk += 1
        _, ref_msg = ref_engine.get_calculated_cpi_info()
        reference_original_cpi_str = ref_msg
    except Exception as e:
        reference_original_cpi_str = f"Erro no cálculo de referência: {e}"

def setup_chosen_simulation(*_):
    global current_sim, history_stack
    history_stack = []

    run_baseline_original_simulation()

    with dashboard_panel_output:
        clear_output()
        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Nenhuma instrução base encontrada na Célula 2. Carregue o programa primeiro.")
            return

        selected = drop_select_stream.value
        if selected == 'original':
            stream = [inst['raw'] for inst in program_instructions]
        elif selected == 'gabarito':
            G_orig = build_dependency_graph(program_instructions)
            no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)
            level_orig = compute_layered_positions(G_orig)

            _, level_pipe, node_text_map_pipe, _, _ = build_pipelined_with_moves_ordered(G_orig, level_orig)

            stages = defaultdict(list)
            for node, lvl in level_pipe.items(): stages[lvl].append(node_text_map_pipe[node])
            stream = []
            for lvl in sorted(stages.keys()):
                for inst in stages[lvl]: stream.append(inst)
            for ctrl in [inst['raw'] for inst in program_instructions if inst['category'] in ('branch', 'jump', 'i_type')]: stream.append(ctrl)
        else:
            if not user_canvas_nodes:
                print("ℹ️ Seu Canvas está em branco. Adicione nós ou selecione 'Gabarito' / 'Original'.")
                return
            canvas_nodes_sorted = sorted(user_canvas_nodes, key=lambda x: x['level'])
            stream = [item['text'] for item in canvas_nodes_sorted]
            for ctrl in [inst['raw'] for inst in program_instructions if inst['category'] in ('branch', 'jump', 'i_type')]: stream.append(ctrl)

        current_sim = TomasuloEngineInteractive(stream)
        btn_step_pc.disabled = False
        btn_final_pc.disabled = False
        render_dashboard_view()

def step_simulation_click(b):
    global current_sim, history_stack
    if current_sim is None or current_sim.is_done(): return

    history_stack.append(copy.deepcopy(current_sim))
    current_sim.step_cycle()
    current_sim.clk += 1
    render_dashboard_view()

def prev_simulation_click(b):
    global current_sim, history_stack
    if not history_stack: return

    current_sim = history_stack.pop()
    btn_step_pc.disabled = False
    btn_final_pc.disabled = False
    render_dashboard_view()

def jump_to_final_click(b):
    global current_sim, history_stack
    if current_sim is None or current_sim.is_done(): return

    with dashboard_panel_output:
        while not current_sim.is_done() and current_sim.clk < 1000:
            history_stack.append(copy.deepcopy(current_sim))
            current_sim.step_cycle()
            current_sim.clk += 1
        render_dashboard_view()

def render_dashboard_view():
    with dashboard_panel_output:
        clear_output(wait=True)
        if current_sim is None: return

        _, current_cpi_str = current_sim.get_calculated_cpi_info()
        print(f"⏱️ CLOCK ATUAL: {current_sim.clk - 1}")
        print(f"📊 DESEMPENHO ATUAL    : {current_cpi_str}")
        print(f"🔄 REFERÊNCIA SEQUENCIAL: {reference_original_cpi_str}\n")

        table_list = []
        for instruction, stages in current_sim.table_data.items():
            display_name = instruction.split('   #')[0]
            row = [display_name] + [stages[h] for h in current_sim.x_headers]
            table_list.append(row)

        headers_display = ["Instruction"] + [h.capitalize() if "_" not in h else h[:h.find("_")].capitalize() + " (It2)" for h in current_sim.x_headers]
        print(tabulate(table_list, headers_display, tablefmt="fancy_grid"))

        if current_sim.is_done():
            print(f"\n🛑 SIMULAÇÃO CONCLUÍDA: O teto limite da 2ª iteração foi alcançado!")
            btn_step_pc.disabled = True
            btn_final_pc.disabled = True

btn_reset_sim.on_click(setup_chosen_simulation)
btn_prev_pc.on_click(prev_simulation_click)
btn_step_pc.on_click(step_simulation_click)
btn_final_pc.on_click(jump_to_final_click)
drop_select_stream.observe(setup_chosen_simulation, names='value')

controls_bar = widgets.HBox([drop_select_stream, btn_reset_sim, btn_prev_pc, btn_step_pc, btn_final_pc], layout=widgets.Layout(gap='10px', margin='10px 0'))
display(controls_bar, dashboard_panel_output)

setup_chosen_simulation()

Output(layout=Layout(margin='14px 0 0 0'))